
# Phase 7 — Prior Case Retrieval: everything from scratch in one notebook (GPU for training, then CPU-side boost)
**Output:** `artifacts/submissions_boost/` — 6 different valid zips (one per technique + the all-runs ensemble) and `index.csv` with dev scores. Each contains its own Phase-1-adapted dense component, so each is a genuine experiment. Compare them by the **Codabench Micro-F1@7**; dev has only 118 queries (±0.01–0.02 noise).

**Inputs to add:** (1) the Phase 7 dataset, (2) your **Phase 1 notebook output** (`models.py`, `config.py`, `artifacts/tokenizer.json`, `artifacts/ckpt_tier2/<arch>_best.pt`). **Nothing from the earlier Phase-7 run is needed.** Accelerator **GPU (T4/P100)**, Internet On.

**Budget (rough, untimed on your data):** ~4–7 h on a T4 — 5 trainings (≤ 60 min each, capped by `budget`), distillation, encoding dev+test with 10 chunks per case (≈ the whole case), then ~30 min of CPU work in Part B. Settings are in §0 (`steps=800`, `eval_chunks=10`, `RUN_ABLATIONS=False`); lower `steps` / `eval_chunks` if you need it shorter. **If the session dies, run again:** finished runs are skipped only while `/kaggle/working/artifacts` still exists, so use *Save Version → Save & Run All* only if the whole run fits Kaggle's session limit (as far as I know ~12 h for GPU notebooks).

Run once with `QUICK = True` (≈ 20–30 min) to check every cell, then set `QUICK = False` and delete `/kaggle/working/artifacts` for the real run.


**v2 (advanced):** Part B now uses 9 lexical scorers (adds LSA + 5-gram shingles), learns the ranker on *mini-worlds* built from the train labels (pools that look like dev/test), and stacks a second transductive ranker on top. Honest note: the gains are not guaranteed — compare with the Codabench score, not dev.


## 0. Settings + dependencies

In [ ]:
QUICK = False          # True  -> tiny budgets, whole notebook in ~20-30 min (smoke test of every cell)
FORCE_RETRAIN = False  # True  -> retrain even if a run's score file already exists (OVERWRITES it!)
FORCE_REBUILD = False  # True  -> re-tokenize the data / recompute TF-IDF even if cached
RUN_ABLATIONS = False  # True -> LoRMA ablations + a no-teacher student for the report (+~1.5-2 h; not needed for the submissions)

# Phase-1 assets are auto-detected in section 1; set these only to override
P1_CKPT = None         # e.g. "/kaggle/input/phase1-out/artifacts/ckpt_tier2/decoder_only_best.pt"
P1_TOKENIZER = None    # e.g. "/kaggle/input/phase1-out/artifacts/tokenizer.json"

if QUICK:
    S = dict(prep_q=40, prep_c=300, chunk_len=256, train_chunks=1, eval_chunks=2, bq=4, enc_bs=16, micro=16,
             steps=20, warm=3, eval_every=0, budget=0, kd_steps=20, abl_steps=15)
else:
    S = dict(prep_q=0, prep_c=0, chunk_len=512, train_chunks=2, eval_chunks=10, bq=8, enc_bs=32, micro=24,
             steps=800, warm=50, eval_every=200, budget=60, kd_steps=1000, abl_steps=300)
# peak learning rates per technique (IA3 needs a much larger LR than the others: it only learns scales)
LR = dict(lorma=5e-4, ia3=3e-3, partial=5e-5, full=2e-5, kd=1e-4)
print("QUICK" if QUICK else "FULL", "settings:", S, "\nLR:", LR)

In [ ]:
# Internet must be On. scikit-learn / tokenizers / torch ship with Kaggle; ijson is only a fallback for >1.5 GB JSON files.
!pip install -q ijson

In [ ]:
import importlib, subprocess
for m in ["torch", "tokenizers", "sklearn", "numpy", "pandas", "ijson"]:
    try:
        mod = importlib.import_module(m); print(f"{m:12s} {getattr(mod, '__version__', 'ok')}")
    except Exception as e:
        print(f"{m:12s} FAILED: {type(e).__name__}: {e}")
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB, compute capability {p.major}.{p.minor} -> autocast {'bf16' if p.major >= 8 else 'fp16'}")
else:
    print("WARNING: no GPU visible -- enable an Accelerator in the notebook settings (CPU is only OK for QUICK smoke tests)")
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)

## 1. Locate the Phase 7 data (§13.3) and the Phase 1 assets

Files are found by name under `/kaggle/input`. Each split's `*_candidates.json` is paired with the `*.json` **in the same folder**, so a pool can only ever be used with its own queries.

In [ ]:
import os, sys, shutil, json

KAGGLE_INPUT = "/kaggle/input"
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
if WORK not in sys.path: sys.path.insert(0, WORK)
print("Available input datasets:", os.listdir(KAGGLE_INPUT) if os.path.exists(KAGGLE_INPUT) else "(none)")

def find_all(base, filename):
    return [os.path.join(r, filename) for r, _, fs in os.walk(base) if filename in fs]

def prefer(paths, words):
    return sorted(paths, key=lambda p: (not any(w in p.lower() for w in words), len(p)))

# ---- Phase 7 data: split -> (queries.json, candidates.json) from the same folder
PCR = {}
for split in ("train", "dev", "test"):
    cands = prefer(find_all(KAGGLE_INPUT, f"{split}_candidates.json"), ("pcr", "phase_7", "phase7"))
    assert cands, f"{split}_candidates.json not found under {KAGGLE_INPUT}"
    qpath = os.path.join(os.path.dirname(cands[0]), f"{split}.json")
    assert os.path.exists(qpath), f"{qpath} missing (expected next to {cands[0]})"
    PCR[split] = (qpath, cands[0])
    print(f"{split:5s} queries {qpath}  ({os.path.getsize(qpath)/1e6:.1f} MB)\n      pool    {cands[0]}  ({os.path.getsize(cands[0])/1e6:.1f} MB)")

# ---- Phase 1 assets
if P1_CKPT is None:
    hits = []
    for name in ("decoder_only_best.pt", "efficient_best.pt", "decoder_only_final.pt", "efficient_final.pt"):
        hits += prefer(find_all(KAGGLE_INPUT, name), ("tier2",))
    assert hits, "No Phase-1 checkpoint found. Add your Phase 1 notebook output as an Input (artifacts/ckpt_tier2/decoder_only_best.pt)."
    P1_CKPT = hits[0]
P1_ROOT = P1_CKPT.split("/artifacts/")[0] if "/artifacts/" in P1_CKPT else os.path.dirname(P1_CKPT)
if P1_TOKENIZER is None:
    tk = prefer(find_all(P1_ROOT, "tokenizer.json") or find_all(KAGGLE_INPUT, "tokenizer.json"), ("artifacts",))
    assert tk, "Phase-1 tokenizer.json not found"
    P1_TOKENIZER = tk[0]
for fn in ("models.py", "config.py"):
    src = prefer(find_all(P1_ROOT, fn) or find_all(KAGGLE_INPUT, fn), ())
    assert src, f"Phase-1 {fn} not found -- add the Phase 1 notebook output (it writes models.py / config.py) as an Input"
    if os.path.abspath(src[0]) != os.path.abspath(os.path.join(WORK, fn)):
        shutil.copy(src[0], os.path.join(WORK, fn))
    print("Phase-1", fn, "<-", src[0])
print("P1_CKPT     :", P1_CKPT, f"({os.path.getsize(P1_CKPT)/1e6:.0f} MB)\nP1_TOKENIZER:", P1_TOKENIZER)

# sanity: the Phase-1 model loads, and it is one we can use as an encoder
import torch
from models import load_checkpoint, count_params
_lm, _cfg, _ck = load_checkpoint(P1_CKPT, "cpu")
print(f"Phase-1 model: {_cfg.architecture}, hidden {_cfg.hidden_size}, {_cfg.num_layers} layers, "
      f"{count_params(_lm)/1e6:.1f}M params, trained {_ck.get('step')} steps, positions {_cfg.max_position_embeddings}")
assert _cfg.architecture in ("decoder_only", "efficient"), "only decoder_only / efficient Phase-1 models can be used as bi-encoders here"
P1_LAYERS, P1_HIDDEN = _cfg.num_layers, _cfg.hidden_size
del _lm
DATA_DIR = "artifacts/pcr_data"
os.makedirs("artifacts", exist_ok=True)

## 2. Write project files

`pcr_lib.py` (data loading, Phase-1 encoder, pooling/chunking, **LoRMA / IA³ / partial / full / student**, score matrices, Micro-F1@7) · `pcr_prepare.py` (tokenize once + TF-IDF matrix) · `pcr_train.py` (one entry point for every method; writes score matrices for dev **and** test).

In [ ]:
%%writefile pcr_lib.py
"""
Phase-7 Prior Case Retrieval (PCR): shared library.

Everything starts from the Phase-1 Legal LM (models.py / config.py / tokenizer / checkpoint from Phase 1).
The LM's causal decoder stack is used as a *bi-encoder*: text -> chunks -> token states -> pooled vector.

  data        : load_split(), SplitData, tf-idf scores (hard negatives + lexical baseline)
  encoder     : lm_hidden() (no LM head), pooling variants (mean / wmean / last / max), chunking
  adaptation  : LoRMALinear (multiplicative low-rank), IA3 scalers, partial / full fine-tuning,
                make_student() (layer-pruned Phase-1 student for knowledge distillation)
  retrieval   : doc embeddings, score matrices (doc-mean and MaxSim aggregation), Micro-F1@7
"""
import json
import math
import os
import pickle
import random
import time
from dataclasses import replace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint  # noqa: F401  (gradient checkpointing in lm_hidden)

SPLITS = ("train", "dev", "test")
POOLS = ("mean", "wmean", "last", "max")
AGGS = ("mean", "maxsim")
K_EVAL = 7


# =============================================================================
# 1. Data loading (robust to {id: rec} / [rec, ...] layouts; text = list of sentences or a string)
# =============================================================================
def _doc_text(rec):
    t = rec.get("text") if isinstance(rec, dict) else rec
    if isinstance(t, (list, tuple)):
        t = " ".join(s if isinstance(s, str) else " ".join(map(str, s)) for s in t)
    return (t or "").strip() if isinstance(t, str) else ""


def _rel_of(rec):
    if not isinstance(rec, dict):
        return None
    for k in ("relevant_candidates", "relevant_cases", "relevant", "labels", "gold"):
        if k in rec and rec[k] is not None:
            v = rec[k]
            if isinstance(v, dict):
                v = list(v.keys())
            return [str(x) for x in v]
    return None


def iter_records(path):
    """Yield (id, record). Streams with ijson for very large files, otherwise plain json."""
    big = os.path.getsize(path) > 1.5e9
    if big:
        try:
            import ijson
            with open(path, "rb") as f:
                c = f.read(1)
                while c.isspace():
                    c = f.read(1)
            with open(path, "rb") as f:
                if c == b"{":
                    for k, v in ijson.kvitems(f, "", use_float=True):
                        yield str(k), v
                else:
                    for i, v in enumerate(ijson.items(f, "item", use_float=True)):
                        yield str(v.get("id", i)), v
            return
        except ImportError:
            pass
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k), v
    else:
        for i, v in enumerate(obj):
            rid = None
            if isinstance(v, dict):
                for k in ("id", "case_id", "doc_id", "query_id", "candidate_id"):
                    if k in v:
                        rid = v[k]
                        break
            yield str(rid if rid is not None else i), v


def load_texts(path):
    """-> ids, texts, rel (dict id -> list of ids, or None when the file has no labels)"""
    ids, texts, rel, has_rel = [], [], {}, False
    for rid, rec in iter_records(path):
        ids.append(rid)
        texts.append(_doc_text(rec))
        r = _rel_of(rec)
        if r is not None:
            has_rel = True
            rel[rid] = r
    return ids, texts, (rel if has_rel else None)


# =============================================================================
# 2. Cached, tokenized split data
# =============================================================================
class SplitData:
    """Tokenized queries + candidate pool of one split (written by pcr_prepare.py)."""

    def __init__(self, data_dir, split):
        with open(os.path.join(data_dir, f"{split}.pkl"), "rb") as f:
            d = pickle.load(f)
        self.split = split
        self.qids, self.cids = d["qids"], d["cids"]
        self.qtok, self.ctok = d["qtok"], d["ctok"]
        self.rel = d["rel"]                      # dict qid -> [cid, ...] or None
        self.meta = d["meta"]
        self.cidx = {c: i for i, c in enumerate(self.cids)}
        tf = os.path.join(data_dir, f"{split}_tfidf.npy")
        self.tfidf = np.load(tf, mmap_mode="r") if os.path.exists(tf) else None
        # positives restricted to the pool (what training can actually use)
        self.pos = []
        if self.rel is not None:
            for q in self.qids:
                self.pos.append(sorted({self.cidx[c] for c in self.rel.get(q, []) if c in self.cidx}))
        self.pad_id, self.bos_id, self.eos_id = d["meta"]["pad_id"], d["meta"]["bos_id"], d["meta"]["eos_id"]


# =============================================================================
# 3. Metrics
# =============================================================================
def topk_ids(S, qids, cids, k=K_EVAL):
    """S: (nq, N) score matrix -> {qid: [k candidate ids]} (a candidate equal to the query id is excluded)."""
    S = np.array(S, dtype=np.float32, copy=True)
    cpos = {c: i for i, c in enumerate(cids)}
    for qi, q in enumerate(qids):
        if q in cpos:
            S[qi, cpos[q]] = -np.inf
    out = {}
    kk = min(k, S.shape[1])
    for qi, q in enumerate(qids):
        part = np.argpartition(-S[qi], kk - 1)[:kk]
        part = part[np.argsort(-S[qi][part], kind="stable")]
        out[q] = [cids[j] for j in part]
    return out


def micro_prf(pred, gold, k=K_EVAL):
    """Micro-Precision/Recall/F1 @k. gold is neither truncated nor padded (spec 13.4)."""
    tp = npred = ngold = 0
    for q, g in gold.items():
        p = pred.get(q, [])[:k]
        tp += len(set(p) & set(g))
        npred += len(p)
        ngold += len(g)
    P = tp / max(npred, 1)
    R = tp / max(ngold, 1)
    F1 = 2 * P * R / (P + R) if P + R > 0 else 0.0
    return P, R, F1


# =============================================================================
# 4. Encoder on top of the Phase-1 LM (no LM head, causal stack, several poolings)
# =============================================================================
def lm_hidden(lm, ids, grad_ckpt=False):
    """Final-norm hidden states (B, T, D) of a Phase-1 decoder_only / efficient LM."""
    from models import rotary_freqs, build_local_window_mask, build_causal_mask
    cfg = lm.cfg
    B, T = ids.shape
    x = lm.drop(lm.tok_emb(ids))
    freqs = rotary_freqs(T, lm.head_dim, ids.device)
    local_mask = full_mask = None
    if cfg.architecture == "efficient":
        local_mask = build_local_window_mask(T, cfg.local_window, ids.device)
        full_mask = build_causal_mask(T, ids.device)
    for i, blk in enumerate(lm.blocks):
        mask = None
        if cfg.architecture == "efficient":
            mask = full_mask if (i + 1) % cfg.global_every == 0 else local_mask
        if grad_ckpt and torch.is_grad_enabled():
            x = torch.utils.checkpoint.checkpoint(blk, x, freqs, mask, use_reentrant=False)
        else:
            x = blk(x, freqs, mask)
    return lm.norm_f(x)


def pool_all(h, mask, names):
    """h (B,T,D) float32, mask (B,T) bool (right padding) -> {name: (B,D)}."""
    out = {}
    B, T, _ = h.shape
    m = mask.unsqueeze(-1).to(h.dtype)
    if "mean" in names:
        out["mean"] = (h * m).sum(1) / m.sum(1).clamp(min=1)
    if "wmean" in names:       # SGPT-style position-weighted mean: later tokens of a causal LM have seen more context
        w = torch.arange(1, T + 1, device=h.device, dtype=h.dtype).view(1, T, 1) * m
        out["wmean"] = (h * w).sum(1) / w.sum(1).clamp(min=1e-6)
    if "last" in names:
        idx = mask.long().sum(1).clamp(min=1) - 1
        out["last"] = h[torch.arange(B, device=h.device), idx]
    if "max" in names:
        out["max"] = h.masked_fill(~mask.unsqueeze(-1), float("-inf")).max(1).values
    return out


def make_chunks(tokens, chunk_len, max_chunks, mode, rng, bos_id, min_tokens=16):
    """Split a token array into <bos>+(chunk_len-1) windows; keep at most max_chunks of them.
    mode 'even'   : evenly spaced windows over the whole stored document (deterministic, used at eval time)
    mode 'random' : random windows (training augmentation)
    A short trailing remainder (< min_tokens) is dropped unless it is the only window."""
    body = chunk_len - 1
    n = len(tokens)
    nseg = max(1, -(-n // body))
    segs = list(range(nseg))
    if nseg > 1 and n - (nseg - 1) * body < min_tokens:
        segs = segs[:-1]
    if len(segs) > max_chunks:
        if mode == "random":
            segs = sorted(rng.sample(segs, max_chunks))
        else:
            segs = sorted(set(np.linspace(0, len(segs) - 1, max_chunks).round().astype(int).tolist()))
    out = []
    for s in segs:
        piece = tokens[s * body:(s + 1) * body]
        out.append(np.concatenate([[bos_id], piece]).astype(np.int64))
    return out


def collate_chunks(chunks, pad_id):
    L = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(chunks), L), dtype=torch.bool)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c)
        mask[i, :len(c)] = True
    return ids, mask


def encode_chunks(lm, chunks, pool_names, device, amp, pad_id, bs=32, grad=False, grad_ckpt=False):
    """List of token arrays -> {pool: (n_chunks, D) L2-normalised float32 tensor}."""
    outs = {p: [] for p in pool_names}
    use_amp = device == "cuda" and amp is not None
    for i in range(0, len(chunks), bs):
        ids, mask = collate_chunks(chunks[i:i + bs], pad_id)
        ids, mask = ids.to(device), mask.to(device)
        with torch.set_grad_enabled(grad):
            with torch.autocast(device_type="cuda", dtype=amp or torch.float16, enabled=use_amp):
                h = lm_hidden(lm, ids, grad_ckpt=grad_ckpt and grad)
            pooled = pool_all(h.float(), mask, pool_names)
        for p in pool_names:
            outs[p].append(F.normalize(pooled[p], dim=-1))
    return {p: torch.cat(v) for p, v in outs.items()}


def doc_mean(E, doc_idx, n_docs):
    """Average chunk embeddings per document and re-normalise. E (Nc,D), doc_idx (Nc,) long tensor."""
    s = torch.zeros(n_docs, E.size(1), device=E.device, dtype=E.dtype).index_add_(0, doc_idx, E)
    return F.normalize(s, dim=-1)


@torch.no_grad()
def encode_corpus(lm, token_list, pool_names, chunk_len, max_chunks, device, amp, pad_id, bos_id, bs=32):
    """-> {pool: (E_chunks (Nc,D) float16 cpu, doc_idx (Nc,) long cpu)}, n_chunks, seconds"""
    rng = random.Random(0)
    chunks, didx = [], []
    for d, toks in enumerate(token_list):
        cs = make_chunks(toks, chunk_len, max_chunks, "even", rng, bos_id)
        chunks.extend(cs)
        didx.extend([d] * len(cs))
    t0 = time.time()
    # length-sort inside the call to cut padding; restore order afterwards
    order = sorted(range(len(chunks)), key=lambda i: len(chunks[i]))
    emb = encode_chunks(lm, [chunks[i] for i in order], pool_names, device, amp, pad_id, bs=bs)
    if device == "cuda":
        torch.cuda.synchronize()
    secs = time.time() - t0
    inv = torch.empty(len(order), dtype=torch.long)
    inv[torch.tensor(order)] = torch.arange(len(order))
    res = {p: (emb[p][inv.to(emb[p].device)].half().cpu(), torch.tensor(didx)) for p in pool_names}
    return res, len(chunks), secs


@torch.no_grad()
def maxsim_matrix(Eq, qidx, nq, Ed, didx, nd, device):
    """Late-interaction aggregation: score(q,d) = mean over q-chunks of (max over d-chunks of cosine)."""
    Eq, Ed = Eq.float().to(device), Ed.float().to(device)
    qidx, didx = qidx.to(device), didx.to(device)
    counts = torch.bincount(didx, minlength=nd)
    maxc = int(counts.max().item())
    Nc = Ed.size(0)
    pad = torch.full((nd, maxc), Nc, dtype=torch.long, device=device)
    order = torch.argsort(didx, stable=True)
    starts = torch.cumsum(counts, 0) - counts
    pos_in_doc = torch.arange(Nc, device=device) - starts[didx[order]]
    pad[didx[order], pos_in_doc] = order
    out = torch.empty(nq, nd, dtype=torch.float32)
    qstarts = torch.cumsum(torch.bincount(qidx, minlength=nq), 0)
    prev = 0
    for q in range(nq):
        end = int(qstarts[q].item())
        sims = Eq[prev:end] @ Ed.T                                      # (nqc, Nc)
        sims = torch.cat([sims, torch.full((sims.size(0), 1), -1e4, device=device)], 1)
        out[q] = sims[:, pad].max(-1).values.mean(0).cpu()              # (nqc, nd, maxc) -> (nd,)
        prev = end
    return out.numpy()


@torch.no_grad()
def score_matrices(qenc, denc, nq, nd, pool_names, device, aggs=AGGS):
    """-> {(pool, agg): (nq, nd) float32 cosine-based score matrix}"""
    res = {}
    for p in pool_names:
        Eq, qi = qenc[p]
        Ed, di = denc[p]
        if "mean" in aggs:
            Dq = doc_mean(Eq.float().to(device), qi.to(device), nq)
            Dd = doc_mean(Ed.float().to(device), di.to(device), nd)
            res[(p, "mean")] = (Dq @ Dd.T).cpu().numpy()
        if "maxsim" in aggs:
            res[(p, "maxsim")] = maxsim_matrix(Eq, qi, nq, Ed, di, nd, device)
    return res


def evaluate_split(lm, data, pool_names, chunk_len, max_chunks, device, amp, bs=32, aggs=AGGS, limit_q=0):
    """Encode queries and pool of one split with the current model -> score matrices + efficiency stats."""
    qn = len(data.qids) if not limit_q else min(limit_q, len(data.qids))
    lm.eval()
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    qenc, nqc, tq = encode_corpus(lm, data.qtok[:qn], pool_names, chunk_len, max_chunks, device, amp,
                                  data.pad_id, data.bos_id, bs)
    denc, ndc, td = encode_corpus(lm, data.ctok, pool_names, chunk_len, max_chunks, device, amp,
                                  data.pad_id, data.bos_id, bs)
    mats = score_matrices(qenc, denc, qn, len(data.cids), pool_names, device, aggs)
    stats = {"chunks_per_s": (nqc + ndc) / max(tq + td, 1e-9), "n_chunks": nqc + ndc, "encode_s": tq + td,
             "peak_mem_gb": (torch.cuda.max_memory_allocated() / 1e9) if device == "cuda" else float("nan")}
    return mats, stats


def dev_metrics(mats, data, qn=None):
    """{(pool, agg): (P, R, F1)} on a labelled split."""
    qids = data.qids[:qn] if qn else data.qids
    gold = {q: data.rel[q] for q in qids}
    return {k: micro_prf(topk_ids(S, qids, data.cids), gold) for k, S in mats.items()}


# =============================================================================
# 5. Adaptation methods
# =============================================================================
class LoRMALinear(nn.Module):
    """LoRMA-style *multiplicative* low-rank adaptation:  W' = (I + s * B A) W0   (A: r x out, B: out x r, B = 0 at init).
    The update is applied to the layer's output:  y = W0 x + s * B(A(W0 x)).  Contrast with additive LoRA:  W0 + B A.
    NOTE: this is the core multiplicative update; the paper's extra rank-inflation tricks are not implemented."""

    def __init__(self, base: nn.Linear, r=16, alpha=16.0):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A.to(h.dtype)), self.B.to(h.dtype)) * self.scale


class IA3Out(nn.Module):
    """IA3: learned element-wise rescaling of a layer's OUTPUT (keys / values)."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        h = self.base(x)
        return h * self.l.to(h.dtype)


class IA3In(nn.Module):
    """IA3: learned element-wise rescaling of a layer's INPUT (the FFN hidden activations feeding w3)."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))


def freeze_all(lm):
    for p in lm.parameters():
        p.requires_grad = False


def make_student(lm, n_layers):
    """Layer-pruned student: keep n_layers evenly spaced Phase-1 blocks (weights copied, embeddings kept).
    Starts from Phase-1 weights, so it satisfies 'start every experiment from the Phase-1 model'."""
    L = len(lm.blocks)
    assert 1 <= n_layers <= L, f"student_layers must be in [1, {L}]"
    idx = sorted(set(np.linspace(0, L - 1, n_layers).round().astype(int).tolist()))
    lm.blocks = nn.ModuleList([lm.blocks[i] for i in idx])
    lm.cfg = replace(lm.cfg, num_layers=len(idx))
    return lm, idx


def apply_method(lm, method, margs):
    """Turn the Phase-1 LM into the trainable retriever for `method`. margs: dict of method hyper-parameters."""
    if method == "zeroshot":
        freeze_all(lm)
    elif method == "full":
        for p in lm.parameters():
            p.requires_grad = True
    elif method == "partial":
        freeze_all(lm)
        k = int(margs.get("partial_layers", 2))
        for blk in list(lm.blocks)[-k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.norm_f.parameters():
            p.requires_grad = True
    elif method == "lorma":
        freeze_all(lm)
        targets = set(str(margs.get("lorma_targets", "q_proj,k_proj,v_proj,o_proj,w1,w2,w3")).split(","))
        r, a = int(margs.get("lorma_rank", 16)), float(margs.get("lorma_alpha", 16))
        for blk in lm.blocks:
            for parent in (blk.attn, blk.ffn):
                for name, child in list(parent.named_children()):
                    if name in targets and isinstance(child, nn.Linear):
                        setattr(parent, name, LoRMALinear(child, r, a))
    elif method == "ia3":
        freeze_all(lm)
        for blk in lm.blocks:
            blk.attn.k_proj = IA3Out(blk.attn.k_proj)
            blk.attn.v_proj = IA3Out(blk.attn.v_proj)
            blk.ffn.w3 = IA3In(blk.ffn.w3)
    elif method == "kd":
        for p in lm.parameters():
            p.requires_grad = True          # student is fully fine-tuned (it is small)
    else:
        raise ValueError(f"unknown method {method}")
    return lm


def set_dropout(lm, p):
    for m in lm.modules():
        if isinstance(m, nn.Dropout):
            m.p = p
    for blk in lm.blocks:
        blk.attn.drop = p


def count_params(lm, trainable_only=False):
    return sum(p.numel() for p in lm.parameters() if (p.requires_grad or not trainable_only))


def load_p1(ckpt_path, device):
    """Phase-1 checkpoint -> (lm, cfg). Only decoder-style Phase-1 models can be used as a bi-encoder here."""
    from models import load_checkpoint
    lm, cfg, _ = load_checkpoint(ckpt_path, device)
    if cfg.architecture not in ("decoder_only", "efficient"):
        raise ValueError(f"Phase-1 architecture '{cfg.architecture}' is not supported as an encoder here; "
                         f"use a decoder_only / efficient Phase-1 checkpoint.")
    return lm, cfg


def build_retriever(p1_ckpt, method, margs, device, dropout=0.0):
    lm, _ = load_p1(p1_ckpt, device)
    if method == "kd":
        lm, kept = make_student(lm, int(margs.get("student_layers", 4)))
        margs = dict(margs, student_kept=kept)
    lm = apply_method(lm, method, margs).to(device)
    set_dropout(lm, dropout)
    return lm, margs


def trainable_state(lm):
    return {n: p.detach().cpu().clone() for n, p in lm.named_parameters() if p.requires_grad}


def save_retriever(path, lm, method, margs, extra=None):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save({"method": method, "margs": margs, "trainable": trainable_state(lm), "extra": extra or {}}, path)


def load_retriever(path, p1_ckpt, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    lm, margs = build_retriever(p1_ckpt, ck["method"], ck["margs"], device)
    missing = lm.load_state_dict(ck["trainable"], strict=False)
    bad = [k for k in ck["trainable"] if k in missing.unexpected_keys]
    assert not bad, f"checkpoint keys not found in rebuilt model: {bad[:5]}"
    lm.eval()
    return lm, ck


# =============================================================================
# 6. Losses
# =============================================================================
def contrastive_loss(eq, ed, pos_col, false_neg_mask, tau, kind="infonce", margin=0.2):
    """eq (B,D), ed (M,D) L2-normalised. pos_col[i] = column of query i's positive.
    false_neg_mask (B,M) True = a *different* relevant candidate of the same query (masked out of the negatives)."""
    cos = eq @ ed.T
    B = eq.size(0)
    if kind == "infonce":
        logits = (cos / tau).masked_fill(false_neg_mask, -1e4)
        return F.cross_entropy(logits, pos_col), logits
    pos = cos[torch.arange(B, device=cos.device), pos_col].unsqueeze(1)
    valid = ~false_neg_mask
    valid[torch.arange(B, device=cos.device), pos_col] = False
    loss = (F.relu(margin - pos + cos) * valid).sum() / valid.sum().clamp(min=1)
    return loss, (cos / tau).masked_fill(false_neg_mask, -1e4)


def kd_loss(student_logits, teacher_logits, T):
    """KL(teacher || student) on temperature-softened in-batch score distributions (Hinton et al.)."""
    pt = F.softmax(teacher_logits / T, dim=-1)
    ls = F.log_softmax(student_logits / T, dim=-1)
    return F.kl_div(ls, pt, reduction="batchmean") * (T * T)

In [ ]:
%%writefile pcr_prepare.py
"""
Prepare the Phase-7 PCR data once (CPU only, no GPU needed):
  * reads <split>.json + <split>_candidates.json  (a candidate pool is ONLY ever used with its own split)
  * tokenizes every query / candidate with the Phase-1 tokenizer (first --store_tokens tokens are kept)
  * computes a TF-IDF cosine matrix (queries x pool): lexical baseline, hard-negative source, hybrid-scoring partner
  * logs the TF-IDF baseline Micro-F1@7 on labelled splits to artifacts/results.jsonl

    python pcr_prepare.py --train_json .../train.json --train_cands .../train_candidates.json ... --tokenizer artifacts/tokenizer.json
"""
import argparse
import json
import os
import pickle
import time

import numpy as np

from pcr_lib import load_texts, topk_ids, micro_prf, SPLITS


def tfidf_matrix(qtexts, ctexts, ngram, max_features):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, ngram), min_df=2, max_df=0.8,
                          max_features=max_features, dtype=np.float32, lowercase=True)
    C = vec.fit_transform(ctexts)                        # vocabulary / idf from the split's own pool (unsupervised)
    Q = vec.transform(qtexts)
    out = np.empty((Q.shape[0], C.shape[0]), dtype=np.float32)
    for i in range(0, Q.shape[0], 256):                  # block-wise: never materialise more than 256 x N dense rows at once
        out[i:i + 256] = (Q[i:i + 256] @ C.T).toarray()
    return out


def main():
    ap = argparse.ArgumentParser()
    for s in SPLITS:
        ap.add_argument(f"--{s}_json", default=None)
        ap.add_argument(f"--{s}_cands", default=None)
    ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--out_dir", default="artifacts/pcr_data")
    ap.add_argument("--store_tokens", type=int, default=8192)
    ap.add_argument("--tfidf_ngram", type=int, default=2)
    ap.add_argument("--tfidf_max_features", type=int, default=300_000)
    ap.add_argument("--limit_queries", type=int, default=0, help="debug: keep only the first N queries per split")
    ap.add_argument("--limit_cands", type=int, default=0, help="debug: keep only the first N candidates per split")
    args = ap.parse_args()

    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tokenizer)
    assert tok.get_vocab_size() < 65536
    pad, bos, eos = (tok.token_to_id(t) for t in ("<pad>", "<bos>", "<eos>"))
    assert None not in (pad, bos, eos), "Phase-1 tokenizer must contain <pad>/<bos>/<eos>"
    os.makedirs(args.out_dir, exist_ok=True)
    char_cap = args.store_tokens * 9

    def encode(texts):
        out = []
        for i in range(0, len(texts), 64):
            encs = tok.encode_batch([t[:char_cap] for t in texts[i:i + 64]], add_special_tokens=False)
            out.extend(np.asarray(e.ids[:args.store_tokens], dtype=np.uint16) for e in encs)
        return out

    for split in SPLITS:
        qp, cp = getattr(args, f"{split}_json"), getattr(args, f"{split}_cands")
        if not qp or not cp:
            print(f"[{split}] not given -- skipped"); continue
        t0 = time.time()
        qids, qtexts, rel = load_texts(qp)
        cids, ctexts, _ = load_texts(cp)
        if args.limit_queries:
            qids, qtexts = qids[:args.limit_queries], qtexts[:args.limit_queries]
        if args.limit_cands:
            keep = set(cids[:args.limit_cands])
            if rel:                                      # debug mode: make sure labelled positives stay in the pool
                for q in qids:
                    keep.update(rel.get(q, []))
            sel = [i for i, c in enumerate(cids) if c in keep]
            cids, ctexts = [cids[i] for i in sel], [ctexts[i] for i in sel]
        if rel is not None:
            rel = {q: rel.get(q, []) for q in qids}
        qtok, ctok = encode(qtexts), encode(ctexts)
        ql, cl = np.array([len(x) for x in qtok]), np.array([len(x) for x in ctok])
        in_pool = None
        if rel is not None:
            cset = set(cids)
            tot = sum(len(v) for v in rel.values())
            in_pool = sum(1 for v in rel.values() for c in v if c in cset) / max(tot, 1)
        meta = {"pad_id": pad, "bos_id": bos, "eos_id": eos, "store_tokens": args.store_tokens,
                "n_queries": len(qids), "n_cands": len(cids),
                "q_tokens_mean": float(ql.mean()), "c_tokens_mean": float(cl.mean()),
                "c_tokens_p90": float(np.percentile(cl, 90)), "frac_c_truncated": float((cl >= args.store_tokens).mean()),
                "frac_gold_in_pool": in_pool,
                "gold_per_query": (float(np.mean([len(v) for v in rel.values()])) if rel else None)}
        print(f"[{split}] {meta}", flush=True)
        S = tfidf_matrix(qtexts, ctexts, args.tfidf_ngram, args.tfidf_max_features)
        np.save(os.path.join(args.out_dir, f"{split}_tfidf.npy"), S)
        with open(os.path.join(args.out_dir, f"{split}.pkl"), "wb") as f:
            pickle.dump({"qids": qids, "cids": cids, "qtok": qtok, "ctok": ctok, "rel": rel, "meta": meta}, f)
        if rel is not None:
            P, R, F1 = micro_prf(topk_ids(S, qids, cids), rel)
            print(f"[{split}] TF-IDF baseline  P@7 {P:.4f}  R@7 {R:.4f}  F1@7 {F1:.4f}", flush=True)
            os.makedirs("artifacts", exist_ok=True)
            with open("artifacts/results.jsonl", "a") as f:
                f.write(json.dumps({"kind": "pcr", "tag": f"tfidf_{split}", "method": "tfidf", "split": split,
                                    "P": P, "R": R, "F1": F1, "params_M": 0.0}) + "\n")
        print(f"[{split}] done in {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pcr_train.py
"""
Fine-tune / adapt the Phase-1 Legal LM for Prior Case Retrieval and write dense score matrices.

  --method zeroshot : Phase-1 LM as-is (how well do Phase-1 representations transfer?)  [not a counted experiment]
  --method lorma    : LoRMA multiplicative low-rank adaptation          (compulsory technique)
  --method ia3      : IA3 scaling vectors on keys / values / FFN        (parameter-efficient)
  --method partial  : only the last K blocks + final norm are trained   (layer-wise fine-tuning)
  --method full     : all parameters trained                            (full fine-tuning)
  --method kd       : knowledge distillation: a layer-pruned Phase-1 student learns from a fine-tuned teacher

Objective: in-batch + TF-IDF hard-negative contrastive loss (InfoNCE or triplet) with false-negative masking.
Outputs: artifacts/pcr_scores/<tag>.npz (dev/test score matrices for every pooling x aggregation),
         artifacts/pcr_runs/<tag>.pt (trainable weights), a row in artifacts/results.jsonl.
"""
import argparse
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn.functional as F

from pcr_lib import (SplitData, build_retriever, apply_method, encode_chunks, doc_mean, make_chunks,
                     evaluate_split, dev_metrics, contrastive_loss, kd_loss, save_retriever, load_retriever,
                     trainable_state, count_params, POOLS, AGGS)


def get_amp_dtype():
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def get_lr(step, lr, warmup, total):
    if step < warmup:
        return lr * (step + 1) / max(1, warmup)
    prog = min(1.0, (step - warmup) / max(1, total - warmup))
    return 0.1 * lr + 0.9 * lr * 0.5 * (1 + math.cos(math.pi * prog))


def build_hard_negs(data, hn_pool, hn_skip):
    """For every training query: TF-IDF-ranked candidates that are NOT labelled relevant (lexically similar but different)."""
    out = []
    for qi in range(len(data.qids)):
        order = np.argsort(-np.asarray(data.tfidf[qi]))[:hn_pool + len(data.pos[qi])]
        pos = set(data.pos[qi])
        out.append([int(j) for j in order[hn_skip:] if int(j) not in pos][:hn_pool])
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["zeroshot", "full", "partial", "lorma", "ia3", "kd"])
    ap.add_argument("--p1_ckpt", required=True)
    ap.add_argument("--data_dir", default="artifacts/pcr_data")
    ap.add_argument("--tag", default=None)
    ap.add_argument("--runs_dir", default="artifacts/pcr_runs")
    ap.add_argument("--scores_dir", default="artifacts/pcr_scores")
    # representation
    ap.add_argument("--pool", default="wmean", choices=POOLS, help="pooling used by the TRAINING loss")
    ap.add_argument("--chunk_len", type=int, default=512)
    ap.add_argument("--train_chunks", type=int, default=2, help="random chunks per document per training step")
    ap.add_argument("--eval_chunks", type=int, default=6, help="max evenly spaced chunks per document at eval time")
    ap.add_argument("--eval_pools", default="mean,wmean,last")
    # optimisation
    ap.add_argument("--steps", type=int, default=300)
    ap.add_argument("--warmup", type=int, default=20)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--wd", type=float, default=0.01)
    ap.add_argument("--batch_queries", type=int, default=8)
    ap.add_argument("--hard_negs", type=int, default=1, help="TF-IDF hard negatives per query (0 = in-batch only)")
    ap.add_argument("--hn_pool", type=int, default=50)
    ap.add_argument("--hn_skip", type=int, default=0, help="skip the top-N TF-IDF hits (likely unlabelled positives)")
    ap.add_argument("--loss", default="infonce", choices=["infonce", "triplet"])
    ap.add_argument("--tau", type=float, default=0.05)
    ap.add_argument("--margin", type=float, default=0.2)
    ap.add_argument("--dropout", type=float, default=0.0)
    ap.add_argument("--grad_clip", type=float, default=1.0)
    ap.add_argument("--micro_chunks", type=int, default=24)
    ap.add_argument("--enc_bs", type=int, default=32)
    ap.add_argument("--grad_ckpt", type=int, default=1)
    ap.add_argument("--eval_every", type=int, default=0, help="dev eval every N steps for checkpoint selection (0 = end only)")
    ap.add_argument("--time_budget_min", type=float, default=0)
    ap.add_argument("--seed", type=int, default=42)
    # method knobs
    ap.add_argument("--lorma_rank", type=int, default=16)
    ap.add_argument("--lorma_alpha", type=float, default=16)
    ap.add_argument("--lorma_targets", default="q_proj,k_proj,v_proj,o_proj,w1,w2,w3")
    ap.add_argument("--partial_layers", type=int, default=2)
    ap.add_argument("--teacher_ckpt", default=None)
    ap.add_argument("--student_layers", type=int, default=4)
    ap.add_argument("--kd_temp", type=float, default=2.0)
    ap.add_argument("--kd_alpha", type=float, default=0.7, help="weight of the KD term vs the hard-label contrastive term")
    ap.add_argument("--kd_emb", type=float, default=1.0, help="weight of embedding-alignment (intermediate representation) loss")
    # debug
    ap.add_argument("--limit_dev_q", type=int, default=0)
    ap.add_argument("--skip_test", type=int, default=0)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = get_amp_dtype()
    use_amp = device == "cuda" and amp is not None
    tag = args.tag or args.method
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    os.makedirs(args.scores_dir, exist_ok=True)
    pools = [p for p in args.eval_pools.split(",") if p]
    if args.pool not in pools:
        pools.append(args.pool)

    margs = dict(lorma_rank=args.lorma_rank, lorma_alpha=args.lorma_alpha, lorma_targets=args.lorma_targets,
                 partial_layers=args.partial_layers, student_layers=args.student_layers)
    lm, margs = build_retriever(args.p1_ckpt, args.method, margs, device, dropout=args.dropout)
    n_total, n_train = count_params(lm), count_params(lm, trainable_only=True)
    print(f"[{tag}] method={args.method} total {n_total/1e6:.2f}M params | trainable {n_train/1e6:.3f}M "
          f"({100*n_train/n_total:.2f}%) | layers {len(lm.blocks)} | device={device} amp={amp}", flush=True)

    dev = SplitData(args.data_dir, "dev")
    assert dev.rel is not None, "dev split must contain relevant_candidates"
    bos, pad = dev.bos_id, dev.pad_id
    qn_dev = args.limit_dev_q or len(dev.qids)

    def dev_eval(pool_names, aggs=("mean",)):
        mats, st = evaluate_split(lm, dev, pool_names, args.chunk_len, args.eval_chunks, device, amp,
                                  bs=args.enc_bs, aggs=aggs, limit_q=args.limit_dev_q)
        return mats, st, dev_metrics(mats, dev, qn_dev)

    best = {"f1": -1.0, "step": 0, "state": None}
    losses, t0, step = [], time.time(), 0

    # ------------------------------------------------------------------ training
    if args.method != "zeroshot":
        train = SplitData(args.data_dir, "train")
        assert train.rel is not None and train.tfidf is not None
        usable = [i for i in range(len(train.qids)) if train.pos[i]]
        print(f"train queries with >=1 in-pool positive: {len(usable)}/{len(train.qids)}", flush=True)
        hard = build_hard_negs(train, args.hn_pool, args.hn_skip) if args.hard_negs > 0 else None

        teacher = None
        if args.method == "kd":
            assert args.teacher_ckpt and os.path.exists(args.teacher_ckpt), "--teacher_ckpt is required for kd"
            teacher, tck = load_retriever(args.teacher_ckpt, args.p1_ckpt, device)
            for p in teacher.parameters():
                p.requires_grad = False
            teacher.eval()
            assert teacher.cfg.hidden_size == lm.cfg.hidden_size
            print(f"teacher: {tck['method']} | {count_params(teacher)/1e6:.2f}M params, {len(teacher.blocks)} layers "
                  f"-> student {n_total/1e6:.2f}M params, {len(lm.blocks)} layers", flush=True)

        params = [p for p in lm.parameters() if p.requires_grad]
        decay = [p for p in params if p.dim() >= 2]
        no_decay = [p for p in params if p.dim() < 2]
        opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.wd}, {"params": no_decay, "weight_decay": 0.0}],
                                lr=args.lr, betas=(0.9, 0.98))
        scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and amp == torch.float16))
        rng = random.Random(args.seed)
        lm.train()
        if args.method == "partial":      # frozen lower blocks stay deterministic (dropout is 0 anyway by default)
            pass

        def doc_embs(model, chunks_all, owners, n_docs, grad):
            with torch.set_grad_enabled(grad):
                enc = encode_chunks(model, chunks_all, [args.pool], device, amp, pad, bs=args.micro_chunks,
                                    grad=grad, grad_ckpt=bool(args.grad_ckpt))[args.pool]
                return doc_mean(enc, torch.tensor(owners, device=device), n_docs)

        def run_eval_point(final=False):
            lm.eval()
            _, _, met = dev_eval([args.pool])
            f1 = met[(args.pool, "mean")][2]
            lm.train()
            print(f"  [dev] step {step} | F1@7 {f1:.4f} (P {met[(args.pool,'mean')][0]:.4f} R {met[(args.pool,'mean')][1]:.4f})", flush=True)
            if f1 > best["f1"]:
                best.update(f1=f1, step=step, state=trainable_state(lm))

        while step < args.steps:
            B = min(args.batch_queries, len(usable))
            qs = rng.sample(usable, B)
            pos_idx = [rng.choice(train.pos[q]) for q in qs]
            hn_idx = []
            if hard is not None:
                for q in qs:
                    cands = hard[q]
                    hn_idx.extend(rng.sample(cands, min(args.hard_negs, len(cands))) if cands else [])
            d_idx = pos_idx + hn_idx
            q_chunks, q_own, d_chunks, d_own = [], [], [], []
            for i, q in enumerate(qs):
                cs = make_chunks(train.qtok[q], args.chunk_len, args.train_chunks, "random", rng, bos)
                q_chunks += cs; q_own += [i] * len(cs)
            for j, c in enumerate(d_idx):
                cs = make_chunks(train.ctok[c], args.chunk_len, args.train_chunks, "random", rng, bos)
                d_chunks += cs; d_own += [j] * len(cs)
            allc, allo = q_chunks + d_chunks, q_own + [B + o for o in d_own]
            # false-negative mask: other relevant candidates of the same query must not act as negatives
            fn = torch.zeros(B, len(d_idx), dtype=torch.bool, device=device)
            for i, q in enumerate(qs):
                ps = set(train.pos[q])
                for j, c in enumerate(d_idx):
                    if j != i and c in ps:
                        fn[i, j] = True
            pos_col = torch.arange(B, device=device)

            lr = get_lr(step, args.lr, args.warmup, args.steps)
            for g in opt.param_groups:
                g["lr"] = lr
            opt.zero_grad(set_to_none=True)
            E = doc_embs(lm, allc, allo, B + len(d_idx), grad=True)
            eq, ed = E[:B], E[B:]
            ce, s_logits = contrastive_loss(eq, ed, pos_col, fn, args.tau, args.loss, args.margin)
            loss = ce
            if teacher is not None:
                Et = doc_embs(teacher, allc, allo, B + len(d_idx), grad=False)
                tq, td = Et[:B], Et[B:]
                t_logits = (tq @ td.T / args.tau).masked_fill(fn, -1e4)
                kd = kd_loss(s_logits, t_logits, args.kd_temp)
                emb = 0.5 * ((1 - (eq * tq).sum(-1)).mean() + (1 - (ed * td).sum(-1)).mean())
                loss = args.kd_alpha * kd + (1 - args.kd_alpha) * ce + args.kd_emb * emb
            if not torch.isfinite(loss):
                raise RuntimeError(f"loss became {loss.item()} at step {step}; lower --lr")
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, args.grad_clip)
            scaler.step(opt); scaler.update()
            step += 1
            losses.append(loss.item())
            if step % 10 == 0 or step == 1:
                print(f"step {step}/{args.steps} | loss {np.mean(losses[-10:]):.4f} | lr {lr:.2e} | {time.time()-t0:.0f}s", flush=True)
            if args.eval_every and step % args.eval_every == 0 and step < args.steps:
                run_eval_point()
            if args.time_budget_min and (time.time() - t0) > args.time_budget_min * 60:
                print(f"time budget of {args.time_budget_min:.0f} min reached at step {step}", flush=True)
                break
        run_eval_point(final=True)
        # restore the best dev checkpoint (model selection uses dev labels only)
        lm.load_state_dict(best["state"], strict=False)
        print(f"best dev F1@7 {best['f1']:.4f} at step {best['step']}", flush=True)
        save_retriever(os.path.join(args.runs_dir, f"{tag}.pt"), lm, args.method, margs,
                       extra={"best_step": best["step"], "args": vars(args)})
        train_minutes = (time.time() - t0) / 60
    else:
        train_minutes = 0.0

    # ------------------------------------------------------------------ final evaluation: every pooling x aggregation
    lm.eval()
    mats_dev, st_dev, met = dev_eval(pools, AGGS)
    save = {f"dev_{p}_{a}": M for (p, a), M in mats_dev.items()}
    if not args.skip_test:
        test = SplitData(args.data_dir, "test")
        mats_te, st_te = evaluate_split(lm, test, pools, args.chunk_len, args.eval_chunks, device, amp,
                                        bs=args.enc_bs, aggs=AGGS)
        save.update({f"test_{p}_{a}": M for (p, a), M in mats_te.items()})
    np.savez_compressed(os.path.join(args.scores_dir, f"{tag}.npz"), **save)

    grid = {f"{p}/{a}": [round(v, 4) for v in prf] for (p, a), prf in met.items()}
    (bp, ba), (bP, bR, bF) = max(met.items(), key=lambda kv: kv[1][2])
    primary = met[(args.pool, "mean")]
    print(f"[{tag}] dev F1@7 by pooling/aggregation: " + json.dumps({k: v[2] for k, v in grid.items()}), flush=True)
    os.makedirs("artifacts", exist_ok=True)
    with open("artifacts/results.jsonl", "a") as f:
        f.write(json.dumps({
            "kind": "pcr", "tag": tag, "method": args.method, "split": "dev",
            "params_M": round(n_total / 1e6, 2), "trainable_M": round(n_train / 1e6, 3), "layers": len(lm.blocks),
            "train_pool": args.pool, "loss": args.loss, "hard_negs": args.hard_negs, "steps": step,
            "best_step": best["step"], "minutes": round(train_minutes, 1),
            "P": primary[0], "R": primary[1], "F1": primary[2],
            "best_cfg": f"{bp}/{ba}", "best_P": bP, "best_R": bR, "best_F1": bF,
            "chunks_per_s": round(st_dev["chunks_per_s"], 1), "peak_mem_gb": round(st_dev["peak_mem_gb"], 2),
            "final_loss": float(np.mean(losses[-10:])) if losses else None, "grid": grid,
        }) + "\n")


if __name__ == "__main__":
    main()

## 3. Helpers — `run()` fails loudly, results tables

In [ ]:
import json, shlex, subprocess, sys, time
import numpy as np, pandas as pd
from IPython.display import display

def run(cmd):
    # Run a command, stream its output, and STOP THE NOTEBOOK if it fails (`!python ...` ignores the exit code).
    args = shlex.split(cmd)
    if args[0] == "python":
        args[0] = sys.executable
    print("$", " ".join(args)[:400], flush=True)
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=WORK)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"[exit {rc} after {(time.time()-t0)/60:.1f} min]", flush=True)
    if rc != 0:
        hint = "  <- SIGKILL: almost certainly out of memory (lower --micro_chunks / --batch_queries)" if rc == -9 else ""
        raise RuntimeError(f"command failed with exit code {rc}{hint}\n{cmd[:300]}")

def load_results():
    p = "artifacts/results.jsonl"
    rows = [json.loads(l) for l in open(p)] if os.path.exists(p) else []
    df = pd.DataFrame(rows)
    if not len(df):
        return df
    df = df.drop_duplicates(subset=["kind", "tag"], keep="last")        # a re-run replaces the old row
    return df[~df.tag.str.startswith("smoke")].reset_index(drop=True)

PCR_COLS = ["tag", "method", "params_M", "trainable_M", "layers", "steps", "minutes", "P", "R", "F1", "best_cfg", "best_F1", "chunks_per_s", "peak_mem_gb"]

def show(prefixes=None, cols=PCR_COLS):
    df = load_results()
    df = df[(df.kind == "pcr") & (df.method != "tfidf")] if len(df) else df
    if not len(df):
        print("(no results yet)"); return
    if prefixes:
        df = df[df.tag.str.startswith(tuple(prefixes))]
    display(df[[c for c in cols if c in df.columns]].dropna(axis=1, how="all").reset_index(drop=True))

# common arguments for every pcr_train.py call
COMMON = (f"--p1_ckpt {P1_CKPT} --data_dir {DATA_DIR} --chunk_len {S['chunk_len']} --train_chunks {S['train_chunks']} "
          f"--eval_chunks {S['eval_chunks']} --batch_queries {S['bq']} --enc_bs {S['enc_bs']} --micro_chunks {S['micro']} "
          f"--eval_every {S['eval_every']} --time_budget_min {S['budget']} --warmup {S['warm']}")

def exp(tag, method, steps=None, extra=""):
    if os.path.exists(f"artifacts/pcr_scores/{tag}.npz") and not FORCE_RETRAIN:
        print(f"[skip] {tag}: artifacts/pcr_scores/{tag}.npz exists (FORCE_RETRAIN=True to redo)"); return
    run(f"python pcr_train.py --method {method} --tag {tag} {COMMON} --steps {steps or S['steps']} --lr {LR.get(method, 5e-4)} {extra}")

## 4. Tokenize once + TF-IDF (§13.3)

Reads each split's queries and **its own** pool, tokenizes with the Phase-1 tokenizer (first 8 192 tokens kept per case), and builds a TF-IDF cosine matrix (queries × pool). TF-IDF is used for (i) a lexical **baseline**, (ii) **hard negatives** (lexically similar but not labelled relevant), (iii) **hybrid scoring** with the dense model. In `QUICK` mode only the first queries / candidates are used.

In [ ]:
if FORCE_REBUILD:
    shutil.rmtree(DATA_DIR, ignore_errors=True)
if os.path.exists(f"{DATA_DIR}/test.pkl") and not FORCE_REBUILD:
    print("tokenized data already built -- skipping (FORCE_REBUILD=True to redo)")
else:
    lim = f"--limit_queries {S['prep_q']} --limit_cands {S['prep_c']}" if QUICK else ""
    run(f"python pcr_prepare.py --tokenizer {P1_TOKENIZER} --out_dir {DATA_DIR} {lim} "
        + " ".join(f"--{s}_json {PCR[s][0]} --{s}_cands {PCR[s][1]}" for s in PCR))

In [ ]:
# Sanity-check the data BEFORE spending GPU time on it
import pickle
rows = []
for s in ("train", "dev", "test"):
    d = pickle.load(open(f"{DATA_DIR}/{s}.pkl", "rb")); rows.append({"split": s, **{k: v for k, v in d["meta"].items() if k not in ("pad_id", "bos_id", "eos_id")}})
meta_df = pd.DataFrame(rows).set_index("split"); display(meta_df.T)
assert meta_df.loc["train", "frac_gold_in_pool"] > 0.9 and meta_df.loc["dev", "frac_gold_in_pool"] > 0.9, \
    "most labelled candidates are NOT in the pool -- check the file pairing / ID format above"
if meta_df.loc["dev", "frac_c_truncated"] > 0.5:
    print("NOTE: >50% of candidates exceed the stored 8192 tokens -- only their first 8192 tokens are encoded (see report: long documents).")
# per-chunk budget actually seen at eval time
print(f"eval encodes up to {S['eval_chunks']} x {S['chunk_len']} = {S['eval_chunks']*S['chunk_len']} tokens per case "
      f"(candidates average {meta_df.loc['dev','c_tokens_mean']:.0f} stored tokens)")
show_tf = load_results(); display(show_tf[show_tf.method == "tfidf"][["tag", "split", "P", "R", "F1"]].reset_index(drop=True))

## 5. Smoke test (a few minutes): every method trains, evaluates and writes scores

Tiny budgets (4 steps, 10 dev queries). If any method is broken it fails **here**, not after hours of training. Smoke rows are excluded from all tables.

In [ ]:
SM = (f"--p1_ckpt {P1_CKPT} --data_dir {DATA_DIR} --chunk_len 256 --train_chunks 1 --eval_chunks 2 --batch_queries 4 --steps 4 "
      f"--warmup 1 --enc_bs 16 --micro_chunks 16 --limit_dev_q 10 --skip_test 1")
run(f"python pcr_train.py --method zeroshot --tag smoke_zeroshot {SM}")
for m in ("lorma", "ia3", "partial", "full"):
    run(f"python pcr_train.py --method {m} --tag smoke_{m} --lr {LR[m]} {SM}")
run(f"python pcr_train.py --method kd --tag smoke_kd --lr {LR['kd']} --teacher_ckpt artifacts/pcr_runs/smoke_full.pt "
    f"--student_layers {max(2, P1_LAYERS // 3)} {SM}")
print("smoke test passed: zeroshot / lorma / ia3 / partial / full / kd all ran")

## 6. Zero-shot: how well do Phase-1 legal representations transfer? (§13.5)

No training at all. One encoder pass gives **every pooling × aggregation** variant (`mean`, position-weighted `wmean`, `last` token × document-mean vs **MaxSim** late interaction), so this is also the pooling / document-representation ablation for the *un-adapted* model.

In [ ]:
exp("zeroshot", "zeroshot")
def grid_table(tag):
    r = load_results(); r = r[(r.kind == "pcr") & (r.tag == tag)].iloc[0]
    g = pd.DataFrame({k: v for k, v in r["grid"].items()}, index=["P@7", "R@7", "F1@7"]).T
    g.index = pd.MultiIndex.from_tuples([tuple(i.split("/")) for i in g.index], names=["pooling", "aggregation"])
    return g
display(grid_table("zeroshot"))
d = load_results(); tf = d[d.tag == "tfidf_dev"].iloc[0]
print(f"TF-IDF baseline on dev:  P@7 {tf.P:.4f}  R@7 {tf.R:.4f}  F1@7 {tf.F1:.4f}")

## 7. The four required adaptation techniques (§13.2.1)

Same data, same objective (in-batch + 1 TF-IDF hard negative per query, InfoNCE with τ = 0.05, other relevant candidates of the same query masked as false negatives), same pooling (`wmean`) — only the **adaptation technique** differs. Dev F1@7 is checked every `eval_every` steps and the best checkpoint is kept.

### Experiment 1 — LoRMA (compulsory)

In [ ]:
exp("exp1_lorma", "lorma")

### Experiment 2 — IA³

In [ ]:
exp("exp2_ia3", "ia3")

### Experiment 3 — Partial (last-K-layers) fine-tuning

In [ ]:
exp("exp3_partial", "partial", extra="--partial_layers 2")

### Experiment 4 — Full fine-tuning

In [ ]:
exp("exp4_full", "full")

In [ ]:
show(["zeroshot", "exp1", "exp2", "exp3", "exp4"])

## 8. Knowledge distillation (mandatory, §13.2.2)

**Teacher** = the best fine-tuned model from §7 (picked automatically by dev F1@7). **Student** = the Phase-1 model with evenly spaced blocks removed (⅓ of the depth by default, Phase-1 weights copied), fully fine-tuned. The student learns from the teacher's

* **score distributions** — KL between softened in-batch query→candidate similarity distributions (`T = 2`),
* **intermediate representations** — cosine alignment of student and teacher case embeddings,
* plus the hard-label contrastive loss (`α = 0.7` on the teacher terms).

`exp5b_student_nokd` is the *same* student trained **without** the teacher (α = 0, no alignment) — the only fair way to say whether distillation actually helped.

In [ ]:
STUDENT_LAYERS = max(2, round(P1_LAYERS / 3))        # override here if you want a different compression ratio
df = load_results()
cand = df[(df.kind == "pcr") & df.tag.isin(["exp1_lorma", "exp2_ia3", "exp3_partial", "exp4_full"])].sort_values("F1", ascending=False)
assert len(cand), "run section 7 first"
TEACHER_TAG = cand.iloc[0].tag
print(f"teacher = {TEACHER_TAG}  (dev F1@7 {cand.iloc[0].F1:.4f}, {cand.iloc[0].params_M} M params, {int(cand.iloc[0].layers)} layers)  ->  student with {STUDENT_LAYERS} layers")
T_CKPT = f"artifacts/pcr_runs/{TEACHER_TAG}.pt"
exp("exp5_kd", "kd", steps=S["kd_steps"], extra=f"--teacher_ckpt {T_CKPT} --student_layers {STUDENT_LAYERS} --kd_temp 2.0 --kd_alpha 0.7 --kd_emb 1.0")
if RUN_ABLATIONS:
    exp("exp5b_student_nokd", "kd", steps=S["kd_steps"], extra=f"--teacher_ckpt {T_CKPT} --student_layers {STUDENT_LAYERS} --kd_alpha 0 --kd_emb 0")
show(["exp5"])

## 9. Ablations on LoRMA (§13.5): negatives, ranking objective, pooling

Each row changes **one** thing relative to `exp1_lorma` (these are analysis runs, not extra techniques): no hard negatives (in-batch only) · 3 harder-to-skip negatives (skip the top-2 TF-IDF hits, which are the likeliest unlabelled positives) · triplet margin loss instead of InfoNCE · mean pooling instead of `wmean` for training.

In [ ]:
if RUN_ABLATIONS:
    A = S["abl_steps"]
    exp("abl_lorma_inbatch_only", "lorma", steps=A, extra="--hard_negs 0")
    exp("abl_lorma_hn3_skip2",    "lorma", steps=A, extra="--hard_negs 3 --hn_skip 2")
    exp("abl_lorma_triplet",      "lorma", steps=A, extra="--loss triplet --margin 0.2")
    exp("abl_lorma_pool_mean",    "lorma", steps=A, extra="--pool mean")
    exp("abl_lorma_ref",          "lorma", steps=A)          # same step budget as the ablations, for a fair reference
    show(["abl_"])

## 10. Everything in one table + pooling × aggregation grid (feeds §13.5)

In [ ]:
print("ALL RUNS (dev, P/R/F1@7 use the training pooling with document-mean aggregation; best_cfg = best pooling/aggregation on dev)")
show()
g = []
r = load_results(); r = r[(r.kind == "pcr") & (r.method != "tfidf")]
for _, row in r.iterrows():
    g.append({"tag": row.tag, **{k: v[2] for k, v in row["grid"].items()}})
print("\nDev F1@7 for every pooling/aggregation variant of every run:")
display(pd.DataFrame(g).set_index("tag").round(4))

## 10b. Quality vs. size / memory / speed (§13.5) — teacher vs. student vs. adapters

In [ ]:
r = load_results(); r = r[(r.kind == "pcr") & (r.method != "tfidf")].copy()
eff = r[["tag", "method", "params_M", "trainable_M", "layers", "minutes", "chunks_per_s", "peak_mem_gb", "F1", "best_F1"]].copy()
display(eff.round(3).reset_index(drop=True))
try:
    t = r[r.tag == TEACHER_TAG].iloc[0]; s = r[r.tag == "exp5_kd"].iloc[0]
    print(f"\nTEACHER {t.tag}: {t.params_M:.1f}M params, {t.chunks_per_s:.0f} chunks/s, {t.peak_mem_gb} GB peak, F1@7 {t.F1:.4f}")
    print(f"STUDENT exp5_kd: {s.params_M:.1f}M params ({100*(1-s.params_M/t.params_M):.0f}% fewer), {s.chunks_per_s:.0f} chunks/s "
          f"({s.chunks_per_s/t.chunks_per_s:.2f}x), {s.peak_mem_gb} GB peak, F1@7 {s.F1:.4f} "
          f"({100*s.F1/max(t.F1,1e-9):.0f}% of the teacher's)")
    n = r[r.tag == "exp5b_student_nokd"]
    if len(n): print(f"same student WITHOUT distillation: F1@7 {n.iloc[0].F1:.4f}  ->  distillation gain {s.F1 - n.iloc[0].F1:+.4f}")
except Exception as e:
    print("student/teacher comparison skipped:", e)

## B0. Settings

In [ ]:
FORCE = FORCE_REBUILD  # recompute cached lexical matrices
TOPK = 200             # stage-1 ranker re-ranks the top-K candidates per query
TOPK2 = 60             # stage-2 ranker re-ranks the top-K of stage 1
N_WORLDS = 2 if QUICK else 6      # train mini-worlds (leave-one-world-out stacking)
N_SEEDS = 2 if QUICK else 4       # bagging: independent world splits, scores averaged
REFIT_WITH_DEV = False            # MUST stay False: spec 13.3.2 allows dev labels only for model selection / local evaluation, NOT for training
WD   = [0.25, 0.5, 1, 2]          # dense weights tried in the fusion (always > 0: every submission contains its own Phase-1-adapted model)
KS   = [3, 5, 10]                 # PRF: number of feedback documents
LAMS = [0.25, 0.5, 1.0]           # PRF: weight
TRANSDUCTIVE = True    # final Sinkhorn uses ALL test queries' scores (no labels). False -> no cross-query step at the end
SK = [(None, 0)] + ([(0.3, 3), (0.15, 10), (0.5, 3)] if TRANSDUCTIVE else [])   # small grid: stage 2 already contains Sinkhorn features
RUNS = ["exp1_lorma", "exp2_ia3", "exp3_partial", "exp4_full", "exp5_kd"]   # one submission each (+ an ensemble)
LIMIT_Q, LIMIT_C = (S['prep_q'], S['prep_c']) if QUICK else (0, 0)     # same subset as pcr_prepare.py so the dense matrices line up
if QUICK: TOPK, TOPK2 = 40, 20
print("QUICK" if QUICK else "FULL", dict(TOPK=TOPK, TOPK2=TOPK2, N_WORLDS=N_WORLDS, WD=WD, KS=KS, LAMS=LAMS, SK=SK))


In [ ]:
import os, sys, json, shutil, time, pickle, zipfile          # (Part B needs these; harmless if already imported)
import numpy as np, pandas as pd
from IPython.display import display
# dense score matrices trained in THIS notebook (sections 6-8)
DENSE = {t: f"artifacts/pcr_scores/{t}.npz" for t in RUNS + ["zeroshot"] if os.path.exists(f"artifacts/pcr_scores/{t}.npz")}
print("dense runs available:", list(DENSE))
assert [r for r in RUNS if r in DENSE], "no trained runs found -- run the experiments in sections 6-8 first"

## B2. Write the library

In [ ]:
%%writefile boost_lib.py
"""
Phase-7 PCR "boost" library: stronger lexical signals + learning-to-rank + pseudo-relevance feedback + fusion with the
dense (Phase-1-adapted) score matrices produced by the first notebook. CPU only.

  data        : load_split()                      (queries + the split's OWN pool, same order as the first notebook)
  lexical     : tfidf_scorer / bm25_matrix / cite_tokens / build_lexical()   -> one (nq x N) score matrix per scorer
  ranking     : zrow, rank_matrix, micro_prf, topk_ids
  learned     : candidate_pairs / pair_features / train_ltr / ltr_matrix      (gradient-boosted trees, trained on TRAIN labels)
  feedback    : prf_scores()                       (Rocchio-style: boost cases similar to the current top-k)
"""
import json
import os
import re

import numpy as np
import scipy.sparse as sp
from scipy.special import logsumexp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

K_EVAL = 7


# ------------------------------------------------------------------------------------------------ data
def _doc_text(rec):
    t = rec.get("text") if isinstance(rec, dict) else rec
    if isinstance(t, (list, tuple)):
        t = " ".join(s if isinstance(s, str) else " ".join(map(str, s)) for s in t)
    return (t or "").strip() if isinstance(t, str) else ""


def _rel_of(rec):
    if not isinstance(rec, dict):
        return None
    for k in ("relevant_candidates", "relevant_cases", "relevant", "labels", "gold"):
        if k in rec and rec[k] is not None:
            v = rec[k]
            return [str(x) for x in (list(v.keys()) if isinstance(v, dict) else v)]
    return None


def load_texts(path):
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    items = obj.items() if isinstance(obj, dict) else [
        (next((v[k] for k in ("id", "case_id", "doc_id") if isinstance(v, dict) and k in v), i), v) for i, v in enumerate(obj)]
    ids, texts, rel, has = [], [], {}, False
    for rid, rec in items:
        ids.append(str(rid)); texts.append(_doc_text(rec))
        r = _rel_of(rec)
        if r is not None:
            has = True; rel[str(rid)] = r
    return ids, texts, (rel if has else None)


class Split:
    """Queries + the split's own candidate pool (+ labels when available)."""

    def __init__(self, name, qpath, cpath, limit_q=0, limit_c=0):
        self.name = name
        self.qids, self.qtexts, self.rel = load_texts(qpath)
        self.cids, self.ctexts, _ = load_texts(cpath)
        if limit_q:
            self.qids, self.qtexts = self.qids[:limit_q], self.qtexts[:limit_q]
        if limit_c:
            keep = set(self.cids[:limit_c])
            if self.rel:
                for q in self.qids:
                    keep.update(self.rel.get(q, []))
            sel = [i for i, c in enumerate(self.cids) if c in keep]
            self.cids, self.ctexts = [self.cids[i] for i in sel], [self.ctexts[i] for i in sel]
        if self.rel is not None:
            self.rel = {q: self.rel.get(q, []) for q in self.qids}
        cidx = {c: i for i, c in enumerate(self.cids)}
        self.pos = [sorted({cidx[c] for c in self.rel[q] if c in cidx}) for q in self.qids] if self.rel else None
        self.qlen = np.array([len(t.split()) for t in self.qtexts], dtype=np.float32)
        self.clen = np.array([len(t.split()) for t in self.ctexts], dtype=np.float32)


# ------------------------------------------------------------------------------------------------ metrics / ranking
def zrow(M):
    M = np.asarray(M, dtype=np.float32)
    return (M - M.mean(1, keepdims=True)) / (M.std(1, keepdims=True) + 1e-9)


def rank_matrix(M):
    order = np.argsort(-M, axis=1)
    rk = np.empty_like(order)
    rk[np.arange(M.shape[0])[:, None], order] = np.arange(M.shape[1])[None, :]
    return rk


def col_z(M):
    """z-score of each candidate's scores ACROSS queries (is this query unusually good for this case?)."""
    return zrow(np.asarray(M, dtype=np.float32).T).T


def sinkhorn_scores(S, tau, iters):
    """Transductive 'each case belongs to ~one query' prior. Alternately normalise rows and columns of exp(S/tau) in the
    log domain; the result's within-row order is the original order corrected for HUBS (cases that score high for many
    queries are pushed to their best query). iters=0 keeps the original order."""
    L = np.asarray(S, dtype=np.float64) / tau
    for _ in range(iters):
        L = L - logsumexp(L, axis=1, keepdims=True)
        L = L - logsumexp(L, axis=0, keepdims=True)
    return L.astype(np.float32)


def topk_ids(S, qids, cids, k=K_EVAL):
    S = np.array(S, dtype=np.float32, copy=True)
    cpos = {c: i for i, c in enumerate(cids)}
    for qi, q in enumerate(qids):
        if q in cpos:
            S[qi, cpos[q]] = -np.inf
    kk = min(k, S.shape[1])
    out = {}
    for qi, q in enumerate(qids):
        part = np.argpartition(-S[qi], kk - 1)[:kk]
        part = part[np.argsort(-S[qi][part], kind="stable")]
        out[q] = [cids[j] for j in part]
    return out


def micro_prf(pred, gold, k=K_EVAL):
    tp = npred = ngold = 0
    for q, g in gold.items():
        p = pred.get(q, [])[:k]
        tp += len(set(p) & set(g)); npred += len(p); ngold += len(g)
    P, R = tp / max(npred, 1), tp / max(ngold, 1)
    return P, R, (2 * P * R / (P + R) if P + R > 0 else 0.0)


def f1_of(M, sp_: "Split"):
    return micro_prf(topk_ids(M, sp_.qids, sp_.cids), {q: sp_.rel[q] for q in sp_.qids})[2]


def prf_of(M, sp_: "Split"):
    return micro_prf(topk_ids(M, sp_.qids, sp_.cids), {q: sp_.rel[q] for q in sp_.qids})


# ------------------------------------------------------------------------------------------------ lexical scorers
def _blocked_dot(Q, D, bs=256):
    out = np.empty((Q.shape[0], D.shape[0]), dtype=np.float32)
    Dt = D.T.tocsr()                       # transpose once, not once per block
    for i in range(0, Q.shape[0], bs):
        out[i:i + bs] = (Q[i:i + bs] @ Dt).toarray()
    return out


def tfidf_scorer(qtexts, ctexts, **kw):
    """Cosine of sublinear TF-IDF vectors; vocabulary / idf are fitted on the split's own pool (unsupervised).
    Returns (score matrix, L2-normalised pool matrix, query matrix). An empty vocabulary gives all-zero scores."""
    try:
        vec = TfidfVectorizer(sublinear_tf=True, dtype=np.float32, **kw)
        D = vec.fit_transform(ctexts)
        Q = vec.transform(qtexts)
    except ValueError:
        z = sp.csr_matrix((len(ctexts), 1), dtype=np.float32)
        return np.zeros((len(qtexts), len(ctexts)), np.float32), z, sp.csr_matrix((len(qtexts), 1), dtype=np.float32)
    return _blocked_dot(Q, D), D, Q


def lsa_scorer(Q, D, n=200, seed=0):
    """Latent-semantic cosine: truncated SVD of the pool's TF-IDF matrix (unsupervised), queries projected into it."""
    from sklearn.decomposition import TruncatedSVD
    n = min(n, min(D.shape) - 1)
    if n < 2:
        return np.zeros((Q.shape[0], D.shape[0]), np.float32)
    svd = TruncatedSVD(n_components=n, algorithm="randomized", n_iter=4, random_state=seed)
    Zd = svd.fit_transform(D)
    Zq = svd.transform(Q)
    Zd /= np.linalg.norm(Zd, axis=1, keepdims=True) + 1e-9
    Zq /= np.linalg.norm(Zq, axis=1, keepdims=True) + 1e-9
    return (Zq @ Zd.T).astype(np.float32)


def shingle_scorer(qtexts, ctexts, n=5):
    """Shared word n-grams (idf-weighted cosine over hashed 5-gram shingles): catches quoted / near-copied passages."""
    from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
    hv = HashingVectorizer(ngram_range=(n, n), n_features=2 ** 22, alternate_sign=False, norm=None, binary=True,
                           lowercase=True, dtype=np.float32)
    D, Q = hv.transform(ctexts), hv.transform(qtexts)
    tf = TfidfTransformer()
    D = tf.fit_transform(D)
    Q = tf.transform(Q)
    return _blocked_dot(Q, D)


def count_matrices(qtexts, ctexts, min_df=2, max_df=0.8):
    vec = CountVectorizer(min_df=min_df, max_df=max_df, dtype=np.float32, lowercase=True)
    Dc = vec.fit_transform(ctexts)
    return vec.transform(qtexts), Dc


def bm25_matrix(Qc, Dc, k1=1.5, b=0.75, qmode="log"):
    """Okapi BM25 between full-text queries (term weights log(1+tf) or binary) and pool documents."""
    N = Dc.shape[0]
    df = np.asarray((Dc > 0).sum(0)).ravel()
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(Dc.sum(1)).ravel()
    avg = max(float(dl.mean()), 1.0)
    Dw = Dc.tocoo(copy=True)
    Dw.data = (Dw.data * (k1 + 1) / (Dw.data + k1 * (1 - b + b * dl[Dw.row] / avg))) * idf[Dw.col]
    Dw = Dw.tocsr()
    Qw = Qc.copy().tocsr()
    Qw.data = np.log1p(Qw.data) if qmode == "log" else np.ones_like(Qw.data)
    return _blocked_dot(Qw, Dw)


_SEC = re.compile(r'\b(?:sections?|secs?\.?|s\.)\s*(\d+[A-Za-z]?(?:\s*(?:,|and|&|/|to)\s*\d+[A-Za-z]?)*)', re.I)
_ART = re.compile(r'\barticles?\s*(\d+[A-Za-z]?(?:\s*(?:,|and|&|/)\s*\d+[A-Za-z]?)*)', re.I)
_NUM = re.compile(r'\d+[A-Za-z]?')
_ACT = re.compile(r'\b((?:[A-Z][A-Za-z&\-]*\s+){1,6}Act)\b(?:,?\s*((?:18|19|20)\d\d))?')


def cite_tokens(text):
    """Statute / article / act mentions as tokens (sec_302, art_21, act_indianpenalcodeact ...): legal 'citation fingerprint'."""
    toks = []
    for m in _SEC.finditer(text):
        toks += ["sec_" + n.lower() for n in _NUM.findall(m.group(1))]
    for m in _ART.finditer(text):
        toks += ["art_" + n.lower() for n in _NUM.findall(m.group(1))]
    for m in _ACT.finditer(text):
        toks.append("act_" + re.sub(r"\W+", "", m.group(1).lower()) + (m.group(2) or ""))
    return " ".join(toks)


def head_tail(text, n_words, tail):
    w = text.split()
    return " ".join(w[-n_words:] if tail else w[:n_words])


LEX_NAMES = ["tfidf_uni", "tfidf_bi", "bm25", "cite", "rare", "head", "tail", "lsa", "shingle5"]


def build_lexical(spl: "Split", bm25_params=(1.5, 0.75, "log"), verbose=True):
    """-> dict name -> (nq, N) float32 matrix, plus Duni (L2-normalised unigram TF-IDF pool matrix, used by PRF)."""
    import time
    q, c = spl.qtexts, spl.ctexts
    out, t0 = {}, time.time()

    def log(n):
        if verbose:
            print(f"  [{spl.name}] {n:10s} done ({time.time()-t0:.0f}s)", flush=True)
    out["tfidf_uni"], Duni, Quni = tfidf_scorer(q, c, ngram_range=(1, 1), min_df=2, max_df=0.8); log("tfidf_uni")
    out["tfidf_bi"], _, _ = tfidf_scorer(q, c, ngram_range=(1, 2), min_df=2, max_df=0.8, max_features=500_000); log("tfidf_bi")
    Qc, Dc = count_matrices(q, c)
    out["bm25"] = bm25_matrix(Qc, Dc, *bm25_params); log("bm25")
    cq, cc = [cite_tokens(t) for t in q], [cite_tokens(t) for t in c]
    out["cite"], _, _ = tfidf_scorer(cq, cc, token_pattern=r"\S+", min_df=1); log("cite")
    out["rare"], _, _ = tfidf_scorer(q, c, ngram_range=(1, 2), min_df=2, max_df=0.02, max_features=400_000); log("rare")
    out["head"], _, _ = tfidf_scorer([head_tail(t, 1500, False) for t in q], [head_tail(t, 1500, False) for t in c], min_df=2, max_df=0.8); log("head")
    out["tail"], _, _ = tfidf_scorer([head_tail(t, 1500, True) for t in q], [head_tail(t, 1500, True) for t in c], min_df=2, max_df=0.8); log("tail")
    out["lsa"] = lsa_scorer(Quni, Duni); log("lsa")
    out["shingle5"] = shingle_scorer(q, c, 5); log("shingle5")
    return out, Duni, (Qc, Dc)


# ------------------------------------------------------------------------------------------------ pseudo-relevance feedback
def prf_scores(F, Duni, k):
    """Rocchio-style feedback in document space: how similar is each case to the CURRENT top-k cases (excluding itself)?"""
    nq, N = F.shape
    k = min(k, N)
    top = np.argpartition(-F, k - 1, axis=1)[:, :k]
    rows = np.repeat(np.arange(nq), k)
    W = sp.csr_matrix((np.full(nq * k, 1.0 / k, dtype=np.float32), (rows, top.ravel())), shape=(nq, N))
    C = W @ Duni
    S = _blocked_dot(C.tocsr(), Duni)
    S[rows, top.ravel()] -= 1.0 / k
    return S


# ------------------------------------------------------------------------------------------------ mini-worlds from TRAIN
def make_worlds(spl: "Split", n_worlds, seed=0, distract=0.08):
    """Split the train queries into groups; a 'world' = (queries of the group, union of THEIR relevant cases + ~8% random
    distractors). Dev / test pools look exactly like this (pool ~ union of the queries' relevant cases, ~1 query per case),
    whereas the full train matrix (827 queries x 4320 cases) does not -- so learning on worlds avoids a train/test shift
    in all the across-query features (column z-scores, query ranks per case, Sinkhorn)."""
    rng = np.random.default_rng(seed)
    nq, N = len(spl.qids), len(spl.cids)
    worlds = []
    for g in np.array_split(rng.permutation(nq), n_worlds):
        g = np.sort(g)
        U = np.unique(np.concatenate([np.asarray(spl.pos[q], dtype=int) for q in g] + [np.empty(0, dtype=int)]))
        others = np.setdiff1d(np.arange(N), U)
        nd = int(distract * len(U))
        extra = rng.choice(others, size=min(nd, len(others)), replace=False) if nd and len(others) else np.empty(0, dtype=int)
        worlds.append((g, np.sort(np.concatenate([U, extra])).astype(int)))
    return worlds


def world_pos(spl, g, d):
    col = {int(x): i for i, x in enumerate(d)}
    return [[col[p] for p in spl.pos[q] if p in col] for q in g]


def world_gold(spl, g, d):
    """{qid: [cids]} restricted to the world (for evaluating a world's matrix)."""
    dset = {spl.cids[int(x)] for x in d}
    return {spl.qids[q]: [c for c in spl.rel[spl.qids[q]] if c in dset] for q in g}


# ------------------------------------------------------------------------------------------------ learning to rank (two stages)
def prepare(Ms, qlen, clen, cap=20):
    """Within-matrix features of every scorer: row z / log row-rank, and the same ACROSS queries (column z / capped rank)."""
    Z = {n: zrow(M) for n, M in Ms.items()}
    return dict(Z=Z, R={n: np.minimum(rank_matrix(M), 200) for n, M in Ms.items()},
                ZC={n: col_z(M) for n, M in Ms.items()}, RC={n: np.minimum(rank_matrix(M.T).T, cap) for n, M in Ms.items()},
                base=np.mean([Z[n] for n in Z], axis=0), qlen=qlen, clen=clen)


def candidate_pairs(base, pos=None, topk=200, n_random=30, seed=0):
    """(qi, dj) pairs: top-k by base score (+ all labelled positives and random negatives when training)."""
    rng = np.random.default_rng(seed)
    nq, N = base.shape
    k = min(topk, N)
    top = np.argpartition(-base, k - 1, axis=1)[:, :k]
    qs, ds = [], []
    for qi in range(nq):
        cand = set(top[qi].tolist())
        if pos is not None:
            cand.update(pos[qi])
            cand.update(rng.choice(N, size=min(n_random, N), replace=False).tolist())
        cand = np.fromiter(cand, dtype=np.int64)
        qs.append(np.full(len(cand), qi)); ds.append(cand)
    return np.concatenate(qs), np.concatenate(ds)


def pair_features(P, qi, dj):
    Z, R, ZC, RC = P["Z"], P["R"], P["ZC"], P["RC"]
    cols = []
    for n in Z:
        cols += [Z[n][qi, dj], np.log1p(R[n][qi, dj]), ZC[n][qi, dj], np.log1p(RC[n][qi, dj])]
    zs = np.stack([Z[n][qi, dj] for n in Z], 1)
    rs = np.stack([R[n][qi, dj] for n in Z], 1)
    zc = np.stack([ZC[n][qi, dj] for n in Z], 1)
    rc = np.stack([RC[n][qi, dj] for n in Z], 1)
    qlen, clen = P["qlen"], P["clen"]
    cols += [zs.mean(1), zs.max(1), zs.min(1), (rs < 20).sum(1).astype(np.float32),
             zc.mean(1), zc.max(1), (rc < 3).sum(1).astype(np.float32), (rc == 0).sum(1).astype(np.float32),
             np.log1p(clen[dj]), np.log1p(qlen[qi]), clen[dj] / np.maximum(qlen[qi], 1.0)]
    return np.stack(cols, 1).astype(np.float32)


def train_ltr(X, y, seed=0, max_iter=250):
    from sklearn.ensemble import HistGradientBoostingClassifier
    return HistGradientBoostingClassifier(max_iter=max_iter, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=40,
                                          l2_regularization=1.0, random_state=seed).fit(X, y)


def _fill_matrix(shape, qi, dj, logit):
    """Per-query z-score of the candidates' log-odds; everything else = (row minimum - 1)."""
    S = np.zeros(shape, dtype=np.float32)
    for q in np.unique(qi):
        m = qi == q
        v = logit[m]
        z = (v - v.mean()) / (v.std() + 1e-9)
        S[q, :] = z.min() - 1.0
        S[q, dj[m]] = z
    return S


def _logit(clf, X):
    p = np.clip(clf.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def stage1_fit(Ps, poss, topk, seed=0):
    X, y = [], []
    for P, pos in zip(Ps, poss):
        qi, dj = candidate_pairs(P["base"], pos, topk, 30, seed)
        ps = [set(p) for p in pos]
        X.append(pair_features(P, qi, dj))
        y.append(np.array([d in ps[q] for q, d in zip(qi, dj)], dtype=int))
    return train_ltr(np.vstack(X), np.concatenate(y), seed)


def stage1_matrix(clf, P, topk):
    qi, dj = candidate_pairs(P["base"], None, topk)
    return _fill_matrix(P["base"].shape, qi, dj, _logit(clf, pair_features(P, qi, dj)))


S2_SK = [(0.3, 3), (0.15, 10)]


def stage2_features(L1, Dw, qlen, clen, qi, dj):
    """Transductive features computed from the stage-1 matrix: row / column ranks, 'this query is the case's best query',
    Sinkhorn-normalised scores, similarity to the current top-5 (PRF), margin to the 7th-best candidate."""
    rk_row = np.minimum(rank_matrix(L1), 200)
    rk_col = np.minimum(rank_matrix(L1.T).T, 20)
    cols = [L1[qi, dj], np.log1p(rk_row[qi, dj]), np.log1p(rk_col[qi, dj]), (rk_col[qi, dj] == 0).astype(np.float32),
            col_z(L1)[qi, dj]]
    for tau, it in S2_SK:
        SK = zrow(sinkhorn_scores(L1, tau, it))
        cols += [SK[qi, dj], np.log1p(np.minimum(rank_matrix(SK), 200)[qi, dj])]
    cols.append(zrow(prf_scores(L1, Dw, 5))[qi, dj])
    m7 = np.sort(L1, axis=1)[:, -min(7, L1.shape[1])]
    cols += [L1[qi, dj] - m7[qi], np.log1p(clen[dj]), np.log1p(qlen[qi])]
    return np.stack(cols, 1).astype(np.float32)


def stage2_fit(L1s, Dws, qlens, clens, poss, topk2, seed=0):
    X, y = [], []
    for L1, Dw, ql, cl, pos in zip(L1s, Dws, qlens, clens, poss):
        qi, dj = candidate_pairs(L1, pos, topk2, 20, seed)
        ps = [set(p) for p in pos]
        X.append(stage2_features(L1, Dw, ql, cl, qi, dj))
        y.append(np.array([d in ps[q] for q, d in zip(qi, dj)], dtype=int))
    return train_ltr(np.vstack(X), np.concatenate(y), seed, max_iter=200)


def stage2_matrix(clf2, L1, Dw, qlen, clen, topk2):
    qi, dj = candidate_pairs(L1, None, topk2)
    return _fill_matrix(L1.shape, qi, dj, _logit(clf2, stage2_features(L1, Dw, qlen, clen, qi, dj)))


## B3. Load the splits (each queries file with **its own** pool)
Same order and subset as `pcr_prepare.py` (IDs in file order), so the dense matrices line up row for row; shapes are checked in §6.

In [ ]:
from boost_lib import *
SPL = {}
for s in ("train", "dev", "test"):
    SPL[s] = Split(s, PCR[s][0], PCR[s][1], LIMIT_Q, LIMIT_C)
    sp_ = SPL[s]
    print(f"{s:5s}: {len(sp_.qids)} queries x {len(sp_.cids)} candidates | mean words q {sp_.qlen.mean():.0f} / cand {sp_.clen.mean():.0f}"
          + (f" | gold/query {np.mean([len(v) for v in sp_.rel.values()]):.1f}" if sp_.rel else ""))
train, dev, test = SPL["train"], SPL["dev"], SPL["test"]

## B4. Nine lexical scorers per split (cached) — and how good is each alone?

In [ ]:
os.makedirs("artifacts/boost", exist_ok=True)
LEX, DUNI, CNT = {}, {}, {}
for s in ("train", "dev", "test"):
    cache = f"artifacts/boost/lex2_{s}{'_quick' if QUICK else ''}.pkl"
    if os.path.exists(cache) and not FORCE:
        LEX[s], DUNI[s], CNT[s] = pickle.load(open(cache, "rb")); print(f"[{s}] loaded cache")
    else:
        t0 = time.time(); LEX[s], DUNI[s], CNT[s] = build_lexical(SPL[s])
        pickle.dump((LEX[s], DUNI[s], CNT[s]), open(cache, "wb")); print(f"[{s}] built in {(time.time()-t0)/60:.1f} min")
rows = []
for n in LEX_NAMES:
    rows.append({"scorer": n, **{f"{s}_F1@7": f1_of(LEX[s][n], SPL[s]) for s in ("train", "dev")}})
display(pd.DataFrame(rows).round(4))

### B4b. Tune BM25 (k1, b, query weighting) on **train + dev**
BM25 has no learned parameters, so tuning it on the labelled splits is ordinary hyper-parameter selection; using train+dev (945 queries) keeps the dev estimate from being over-fitted.

In [ ]:
res = []
for k1 in (0.9, 1.5, 3.0):
    for b in (0.5, 0.8, 1.0):
        f = [f1_of(bm25_matrix(*CNT[s], k1, b, "log"), SPL[s]) for s in ("train", "dev")]
        res.append(dict(k1=k1, b=b, qmode="log", train=f[0], dev=f[1], avg=(f[0] * len(train.qids) + f[1] * len(dev.qids)) / (len(train.qids) + len(dev.qids))))
best = max(res, key=lambda r: r["avg"])
for qm in ("bin",):
    f = [f1_of(bm25_matrix(*CNT[s], best["k1"], best["b"], qm), SPL[s]) for s in ("train", "dev")]
    res.append(dict(k1=best["k1"], b=best["b"], qmode=qm, train=f[0], dev=f[1], avg=(f[0] * len(train.qids) + f[1] * len(dev.qids)) / (len(train.qids) + len(dev.qids))))
display(pd.DataFrame(res).sort_values("avg", ascending=False).round(4).head(8))
best = max(res, key=lambda r: r["avg"])
print("BM25 chosen:", {k: best[k] for k in ("k1", "b", "qmode")})
for s in ("train", "dev", "test"):
    LEX[s]["bm25"] = bm25_matrix(*CNT[s], best["k1"], best["b"], best["qmode"])
print({s: round(f1_of(LEX[s]["bm25"], SPL[s]), 4) for s in ("train", "dev")}, "<- BM25 F1@7 (train, dev)")

## B5. Two-stage learning-to-rank on **mini-worlds** built from the TRAIN labels, selected on dev
*Why worlds?* In dev/test the pool is roughly the union of the queries' relevant cases (about one query per case). The full train matrix (827 queries × 4320 cases) is looser, so cross-query features (is this query the best one for that case?) behave differently there. A *world* = a group of train queries + the union of their relevant cases + ~8% distractors, which looks like dev/test.
Everything is **bagged over several world splits**. Rankers are trained on TRAIN labels only; dev labels are used only for model selection and evaluation (spec 13.3.2).
**Stage 1** (per-scorer z-scores / ranks, within and across queries) is trained on worlds. **Stage 2** is trained on *out-of-fold* stage-1 scores (leave-one-world-out) and adds transductive features: Sinkhorn-normalised scores, column ranks, similarity to the current top-5, margin to the 7th candidate. Only TRAIN labels are used; dev just selects which stage to use downstream.

In [ ]:
for s in ("train", "dev"):
    slots = sum(len(v) for v in SPL[s].rel.values())
    print(f"{s}: {slots} relevant pairs / {len(SPL[s].cids)} pool cases = {slots/len(SPL[s].cids):.2f} per case")
est = len(test.qids) * np.mean([len(v) for v in dev.rel.values()]) / len(test.cids)
print(f"test: ~{est:.2f} relevant pairs per pool case (estimate from dev's gold/query)")

PX = {s: prepare(LEX[s], SPL[s].qlen, SPL[s].clen) for s in ("dev", "test")}
BASE = {s: PX[s]["base"] for s in PX}
Q_, C_ = {s: SPL[s].qlen for s in PX}, {s: SPL[s].clen for s in PX}
L1d, L2d, L1t, L2t, L1t_r, L2t_r = [], [], [], [], [], []      # per-seed matrices (dev / test / test re-fitted with dev labels)
for seed in range(N_SEEDS):
    WORLDS = make_worlds(train, N_WORLDS, seed=seed)
    PW, POSW, DWW = [], [], []
    for g, d in WORLDS:
        Ms = {n: LEX["train"][n][np.ix_(g, d)] for n in LEX_NAMES}
        PW.append(prepare(Ms, train.qlen[g], train.clen[d])); POSW.append(world_pos(train, g, d)); DWW.append(DUNI["train"][d])
    L1W = []
    for w in range(len(WORLDS)):
        tr = [i for i in range(len(WORLDS)) if i != w] or [w]
        L1W.append(stage1_matrix(stage1_fit([PW[i] for i in tr], [POSW[i] for i in tr], TOPK, seed), PW[w], TOPK))
    ql, cl = [p["qlen"] for p in PW], [p["clen"] for p in PW]
    clf1 = stage1_fit(PW, POSW, TOPK, seed)
    clf2 = stage2_fit(L1W, DWW, ql, cl, POSW, TOPK2, seed)
    a1 = {s: stage1_matrix(clf1, PX[s], TOPK) for s in PX}
    a2 = {s: stage2_matrix(clf2, a1[s], DUNI[s], Q_[s], C_[s], TOPK2) for s in PX}
    L1d.append(a1["dev"]); L2d.append(a2["dev"]); L1t.append(a1["test"]); L2t.append(a2["test"])
    if REFIT_WITH_DEV:      # dev is a real, test-shaped world; its stage-1 scores are out-of-fold (clf1 never saw dev)
        clf1r = stage1_fit(PW + [PX["dev"]], POSW + [dev.pos], TOPK, seed)
        b1 = stage1_matrix(clf1r, PX["test"], TOPK)
        clf2r = stage2_fit(L1W + [a1["dev"]], DWW + [DUNI["dev"]], ql + [Q_["dev"]], cl + [C_["dev"]], POSW + [dev.pos], TOPK2, seed)
        L1t_r.append(b1); L2t_r.append(stage2_matrix(clf2r, b1, DUNI["test"], Q_["test"], C_["test"], TOPK2))
    print(f"seed {seed}: dev F1 stage1 {f1_of(a1['dev'], dev):.4f} | stage2 {f1_of(a2['dev'], dev):.4f}", flush=True)

L1 = {"dev": zrow(np.mean(L1d, 0)), "test": zrow(np.mean(L1t, 0))}
L2 = {"dev": zrow(np.mean(L2d, 0)), "test": zrow(np.mean(L2t, 0))}
LTR = L1
cands = {"zsum": zrow(BASE["dev"]), "stage1": L1["dev"], "stage2": L2["dev"]}
f_each = {k: f1_of(v, dev) for k, v in cands.items()}
print(f"dev F1@7 (bagged over {N_SEEDS} seeds):", {k: round(v, 4) for k, v in f_each.items()})
LEXMODE = max(f_each, key=f_each.get)
LEXS = {"dev": cands[LEXMODE], "test": {"zsum": zrow(BASE["test"]), "stage1": L1["test"], "stage2": L2["test"]}[LEXMODE]}
if REFIT_WITH_DEV and LEXMODE != "zsum":      # the TEST ranker also learns from the labelled dev pool (settings above were chosen without it)
    LEXS["test"] = zrow(np.mean(L1t_r if LEXMODE == "stage1" else L2t_r, 0))
print("lexical base used downstream:", LEXMODE, "| test ranker re-fitted with dev labels:", REFIT_WITH_DEV and LEXMODE != "zsum")


## B6. Dense (Phase-1-adapted) scores trained above (§6–§8)
For each run the pooling / aggregation variant with the best **dense-only** dev F1 is used; the shapes are checked against the splits loaded here.

In [ ]:
DZ = {}      # tag -> dict(key, dev z-matrix, test z-matrix, dense-only dev F1)
for tag, path in DENSE.items():
    z = np.load(path)
    keys = [k for k in z.files if k.startswith("dev_")]
    ok = [k for k in keys if z[k].shape == (len(dev.qids), len(dev.cids)) and z["test_" + k[4:]].shape == (len(test.qids), len(test.cids))]
    if not ok:
        print(f"[skip] {tag}: matrix shape {z[keys[0]].shape} != ({len(dev.qids)}, {len(dev.cids)}) -- from a QUICK / different run?"); continue
    scores = {k: f1_of(z[k], dev) for k in ok}
    kb = max(scores, key=scores.get)
    DZ[tag] = dict(key=kb[4:], dev=zrow(z[kb]), test=zrow(z["test_" + kb[4:]]), f1=scores[kb])
rows = [{"run": t, "best dense cfg": v["key"], "dense-only dev F1": v["f1"]} for t, v in DZ.items()]
display(pd.DataFrame(rows).round(4))
ENS_TAGS = [t for t in RUNS if t in DZ]
if len(ENS_TAGS) >= 2:
    DZ["ENSEMBLE"] = dict(key="mean of " + "+".join(ENS_TAGS), dev=zrow(np.mean([DZ[t]["dev"] for t in ENS_TAGS], 0)),
                          test=zrow(np.mean([DZ[t]["test"] for t in ENS_TAGS], 0)), f1=None)
    DZ["ENSEMBLE"]["f1"] = f1_of(DZ["ENSEMBLE"]["dev"], dev)
    print("ensemble of", ENS_TAGS, "dense-only dev F1:", round(DZ["ENSEMBLE"]["f1"], 4))

## B7. Fusion + pseudo-relevance feedback, tuned per run on dev
For each run: `F = z(lexical) + w·z(dense)`, then `F ← z(Sinkhorn_τ,it(F))` (optional), `final = z(F) + λ·z(PRF_k(F))`; `(w, τ, it, k, λ)` is chosen on dev from the coarse grids in §0 in two stages (w × Sinkhorn first, then PRF) (`w` is always > 0 so every submission contains its own Phase-1-adapted model; `λ = 0` = no PRF).

In [ ]:
def fuse(cfg, zd, split):
    F = zrow(LEXS[split] + (cfg["wd"] * zd if cfg["wd"] else 0))
    if cfg["sk_it"]:
        F = zrow(sinkhorn_scores(F, cfg["tau"], cfg["sk_it"]))
    if cfg["k"]:
        F = zrow(F + cfg["lam"] * zrow(prf_scores(F, DUNI[split], cfg["k"])))
    return F

def tune(zd_dev):
    # stage A: dense weight x Sinkhorn setting; stage B: PRF on top of the best of A (keeps the search small and less over-fitted)
    best = dict(f1=-1)
    for wd in (WD if zd_dev is not None else [0]):
        for tau, it in SK:
            cfg = dict(wd=wd, tau=tau or 1.0, sk_it=it, k=0, lam=0.0)
            f = f1_of(fuse(cfg, zd_dev, "dev"), dev)
            if f > best["f1"] + 1e-9: best = dict(cfg, f1=f)
    baseA = dict(best)
    for k in KS:
        for lam in LAMS:
            cfg = dict(baseA, k=k, lam=lam)
            f = f1_of(fuse(cfg, zd_dev, "dev"), dev)
            if f > best["f1"] + 1e-9: best = dict(cfg, f1=f)
    return best

def apply(cfg, zd, split="test"):
    return fuse(cfg, zd, split)

lex_only_dev = f1_of(LEXS["dev"], dev)
print(f"lexical (+LTR) only, dev F1 {lex_only_dev:.4f}")
CFG, rows = {}, []
for tag in [r for r in RUNS if r in DZ] + (["ENSEMBLE"] if "ENSEMBLE" in DZ else []):
    t0 = time.time(); c = tune(DZ[tag]["dev"]); CFG[tag] = c
    P, Rr, F1 = prf_of(apply(c, DZ[tag]["dev"], "dev"), dev)
    rows.append({"run": tag, "dense_cfg": DZ[tag]["key"], "w_dense": c["wd"], "sk_tau": c["tau"] if c["sk_it"] else None, "sk_iters": c["sk_it"], "prf_k": c["k"], "prf_lambda": c["lam"], "dev_P": P, "dev_R": Rr, "dev_F1": F1})
    print(f"  {tag:14s} tuned in {time.time()-t0:.0f}s -> dev F1 {F1:.4f}")
if not rows:       # no dense files: lexical-only fallback (contains NO Phase-1 model -> a reference only, not a valid experiment)
    c = tune(None); CFG["LEXICAL_ONLY"] = c
    P, Rr, F1 = prf_of(fuse(c, None, "dev"), dev)
    rows.append({"run": "LEXICAL_ONLY", "dense_cfg": None, "w_dense": 0, "sk_tau": c["tau"] if c["sk_it"] else None, "sk_iters": c["sk_it"], "prf_k": c["k"], "prf_lambda": c["lam"], "dev_P": P, "dev_R": Rr, "dev_F1": F1})
    print(f"lexical-only (no Phase-1 model!) tuned -> dev F1 {F1:.4f}")
TAB = pd.DataFrame(rows).sort_values("dev_F1", ascending=False).reset_index(drop=True)
display(TAB.round(4))

## B8. What helped? Dev F1 stage by stage (for the report)

In [ ]:
stage = [("TF-IDF unigram (first notebook's baseline family)", f1_of(LEX["dev"]["tfidf_uni"], dev)),
         ("TF-IDF 1-2gram", f1_of(LEX["dev"]["tfidf_bi"], dev)),
         ("BM25 (tuned)", f1_of(LEX["dev"]["bm25"], dev)),
         ("equal-weight z-sum of 9 lexical scorers", f1_of(BASE["dev"], dev)),
         ("stage-1 LTR on train worlds", f1_of(L1["dev"], dev)),
         ("stage-2 stacked transductive LTR", f1_of(L2["dev"], dev))]
for t, v in DZ.items():
    stage.append((f"dense only: {t} ({v['key']})", v["f1"]))
for _, r in TAB.iterrows():
    stage.append((f"FINAL {r.run}: lexical-LTR + {r.w_dense}*dense, Sinkhorn(tau={r.sk_tau}, it={r.sk_iters}), PRF(k={r.prf_k}, lam={r.prf_lambda})", r.dev_F1))
display(pd.DataFrame(stage, columns=["stage", "dev F1@7"]).round(4))

## B9. Write the submissions (one zip per run + the ensemble) and validate them

In [ ]:
OUT = "artifacts/submissions_boost"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
pool = set(test.cids); idx = []; preds = {}
NAMES = {"exp1_lorma": "boost_exp1_lorma", "exp2_ia3": "boost_exp2_ia3", "exp3_partial": "boost_exp3_partial",
         "exp4_full": "boost_exp4_full", "exp5_kd": "boost_exp5_kd", "ENSEMBLE": "boost_ensemble_all_runs",
         "LEXICAL_ONLY": "REFERENCE_lexical_only_NO_phase1_model"}
for tag, c in CFG.items():
    F = fuse(c, DZ[tag]["test"] if tag in DZ else None, "test")
    pr = topk_ids(F, test.qids, test.cids)
    assert set(pr) == set(test.qids)
    for q, ids in pr.items():
        assert len(ids) == 7 and len(set(ids)) == 7 and all(x in pool for x in ids), f"{tag}/{q}"
    name = NAMES[tag]; os.makedirs(f"{OUT}/{name}")
    json.dump(pr, open(f"{OUT}/{name}/prediction.json", "w"), indent=2)
    with zipfile.ZipFile(f"{OUT}/{name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(f"{OUT}/{name}/prediction.json", arcname="prediction.json")
    preds[name] = pr
    r = TAB[TAB.run == tag].iloc[0]
    idx.append(dict(submission=f"{name}.zip", dense_run=tag, dense_cfg=(DZ[tag]["key"] if tag in DZ else None), w_dense=c["wd"], sk_tau=(c["tau"] if c["sk_it"] else None), sk_iters=c["sk_it"], prf_k=c["k"], prf_lambda=c["lam"],
                    dev_P=r.dev_P, dev_R=r.dev_R, dev_F1=r.dev_F1))
IDX = pd.DataFrame(idx).sort_values("dev_F1", ascending=False).reset_index(drop=True)
if len(IDX):
    IDX.to_csv(f"{OUT}/index.csv", index=False)
    shutil.copy(f"{OUT}/{IDX.submission[0]}", f"{OUT}/boost_best_dev.zip")
display(IDX.round(4))
names = list(preds)
ov = pd.DataFrame([[np.mean([len(set(preds[a][q]) & set(preds[b][q])) / 7 for q in test.qids]) for b in names] for a in names], index=names, columns=names)
print("Mean top-7 overlap between the submissions (1.0 = identical):"); display(ov.round(2))
if "LEXICAL_ONLY" in CFG:
    print("WARNING: no dense score files were found, so the only zip is a lexical-only REFERENCE. It contains no Phase-1 model and does not count as one of the required experiments.")
print("Upload to Codabench, one at a time (best dev first):"); [print("  " + OUT + "/" + f) for f in IDX.submission]

## B10. Notes for the report
* **Technique mapping:** each zip fuses a *different Phase-1-adapted dense run* (LoRMA, IA³, partial FT, full FT, distilled student) with the shared two-stage lexical LTR (+ final Sinkhorn/PRF) pipeline, so the five technique submissions stay distinct and each still starts from the Phase-1 LM; `boost_ensemble_all_runs` averages their dense scores.
* **What to read from §8:** the gap between *dense only* and *FINAL* shows how much of the score comes from lexical signals / re-ranking vs. the Phase-1 encoders; report it honestly (a hybrid is allowed, but say that most of the gain is lexical if the table shows that).
* **Caveats:** weights `(w, k, λ)` are tuned on 118 dev queries, so differences of ~0.01 between runs are within noise; judge by the Codabench Micro-F1@7. LTR uses only TRAIN labels; BM25 parameters use train+dev labels (no learned parameters).